# Uncertainty Quantification for LLMs — Tutorial Notebook



This notebook is a hands-on tutorial for **Uncertainty Quantification (UQ)** in Large Language Models.

> *UQ asks: "How confident should we be in this model's answer?"*  
> A well-calibrated uncertainty score correlates with model errors — high uncertainty → likely wrong answer.

---

We demonstrate every major UQ family using two complementary libraries on **medical question-answering** examples (clinical prompts, MedGemma model).

**Domain:** Clinical NLP (MedGemma / configurable)  
**Libraries:** [`lm-polygraph`](https://github.com/IINemo/lm-polygraph) · [`uqlm`](https://github.com/cvs-health/uqlm)

---

## Notebook Structure

| Section | Family | Access | Key Methods |
|---|---|---|---|
| **1. Verbalized** | Black-box | Text only | Verbalized 1S/2S, Linguistic, P(True) |
| **2. Consistency-based** | Black-box | Text only | EigLaplacian, SemanticEntropy, SAR, SemanticDensity, LexicalSimilarity |
| **2.6 UQLM Scorers** | Black-box | Text only | Entailment, CosineSim, ExactMatch, SemanticNegentropy |
| **3. Claim-Level UQ** | Black-box | Text only | LongTextUQ, FrequencyScoring, CoCoA |
| **4. Library Comparison** | — | — | lm-polygraph vs uqlm side-by-side |
| **5. Summary Dashboard** | — | — | All methods heatmap |
| **White-box (Introspective)** | White-box | Logits + hidden states + attention | AttentionScore, Mahalanobis, RelativeMahalanobis, EigenScore, RAUQ, Focus |
| **Advanced (Reasoning / TTS)** | Black/White-box | Sequence + Step | Best-of-N, Uncertainty Gate for Tool Use, Entropy-Gated Branching, DEER |
| **Multimodal** | Black/White-box | Sequence-level only | See companion notebook `Multimodal+Normalization.ipynb` |
| **Normalization** | — | — | Min-max, quantile, isotonic PCC |

##### Environment setup

**Local runs:** copy `.env.example` to `.env` at the project root and edit your settings:

```bash
cp .env.example .env
# then edit .env — PROVIDER, MODEL, MODE, and API keys
```

The notebook loads `.env` automatically.

**Google Colab:** use Colab Secrets for `HF_TOKEN`, `OPENAI_API_KEY`, and optionally `PROVIDER` / `MODEL` / `MODE`.

In [ ]:
%pip install -q lm-polygraph uqlm transformers accelerate langchain langchain-huggingface langchain_openai python-dotenv

##### Load  the requested libraries

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import torch
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer

## Model selection
### Setup the Control Panel
In this first phase, we lay the foundations for our *Uncertainty Quantification* experiment. Configuration is driven by your project **`.env` file** (copy from `.env.example`) so you can reconfigure the notebook without editing code.

**1. Imports (The Wrappers)**
* We import the `BlackboxModel` and `WhiteboxModel` classes for the `lm_polygraph` library.
* We import the *Scorers* for the `uqlm` library, alongside the **LangChain** adapters (`ChatOpenAI`, `ChatHuggingFace`). LangChain acts as a universal bridge, allowing `uqlm` to communicate with any external API using a standard format.

**2. Configuration Variables (in `.env`)**
* `PROVIDER`: The main switch. Choose `"openai"` or `"huggingface"` to route all requests to the respective servers.
  > 💡 **Why only these two providers?** While `uqlm` (thanks to LangChain) supports dozens of API providers, `lm_polygraph` currently only supports OpenAI and Hugging Face for Black/Grey-box inference. We deliberately restricted the scope to these two providers to ensure a rigorously fair, 1:1 comparative analysis between the two libraries.
* `MODEL`: The exact name of the model you want to test (e.g., `"google/gemma-2-2b-it"`, `"gpt-3.5-turbo"` or `"google/medgemma-4b-it"`).
* `MODE`: Choose `"white"` to download the model's weights into local memory (VRAM), unlocking attention matrices and hidden states, or `"black"` to query the model via API. This applies to both `lm_polygraph` and `uqlm` in this notebook.

> ⚠️ **Crucial Note on UQ Availability and Access Levels:** The mode you select directly dictates the arsenal of Uncertainty Quantification techniques at your disposal. This is a strict hierarchy:
> * **White-Box Mode (Total Access):** By downloading the weights locally, you gain complete mathematical access to the model's internals. Because you have the highest level of access, a White-box model can execute **ALL** UQ methods (White-box and Black-box).
> * **Black-Box Mode (Restricted Access):** This restricts access purely to the external API outputs. Consequently, you are limited **ONLY** to Black-box statistical methods (like Self-Consistency or text-based semantic similarity). You cannot apply attention-based or entropy-based methods here unless the API explicitly supports probability extraction (Grey-box).

### Sanity Checks & Secure Authentication
Before allocating heavy resources or initiating network requests, this block loads your **`.env` file**, validates the configuration, and verifies API credentials.

**1. Architectural Sanity Checks**
Not all configurations are physically possible. For instance, OpenAI models (like GPT-4 or GPT-3.5) are proprietary and closed-source. It is impossible to download their weights into your local VRAM. If you set `PROVIDER=openai` and `MODE=white` in `.env`, the notebook raises a clear `ValueError`.

**2. Configuration & credentials from `.env`**
* `PROVIDER`, `MODEL`, and `MODE` are read from `.env` (see `.env.example`). After provider and model are set, `load_config_from_env()` loads and validates the matching API credential (`HF_TOKEN` or `OPENAI_API_KEY`).
* Secrets are never printed — only confirmation messages appear in the output.


In [ ]:
def _find_env_file() -> Path | None:
    """Locate project-root .env (works when cwd is repo root or notebooks/)."""
    roots = [Path.cwd()]
    if Path.cwd().name == "notebooks":
        roots.append(Path.cwd().parent)
    else:
        roots.extend([Path.cwd().parent, Path.cwd() / "notebooks"])
    seen: set[Path] = set()
    for root in roots:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        env_file = root / ".env"
        if env_file.is_file():
            return env_file
    return None


def _load_dotenv_file() -> Path | None:
    """Load .env into os.environ. Safe to call multiple times."""
    env_file = _find_env_file()
    if env_file is None:
        return None
    load_dotenv(env_file, override=True)
    return env_file


def _env(name: str, default: str = "") -> str:
    """Read a stripped environment variable; treat blank values as missing."""
    value = os.environ.get(name, default)
    if value is None:
        return default
    value = value.strip()
    return value if value else default


def set_config() -> dict[str, str]:
    """Read PROVIDER, MODEL, and MODE from .env / environment; prompt if missing."""
    _load_dotenv_file()

    provider = _env("PROVIDER")
    if not provider or provider.lower() not in {"openai", "huggingface"}:
        while True:
            provider = input("Select PROVIDER ('openai' or 'huggingface'): ").strip().lower()
            if provider in {"openai", "huggingface"}:
                os.environ["PROVIDER"] = provider
                break
            print("Invalid input. Please type exactly 'openai' or 'huggingface'.")
    else:
        provider = provider.lower()

    model = _env("MODEL")
    if not model:
        while True:
            model = input("Enter MODEL (Examples: google/gemma-2-2b-it, google/medgemma-4b-it, gpt-3.5-turbo):").strip()
            if model:
                os.environ["MODEL"] = model
                break
            print("MODEL cannot be empty.")

    if provider == "openai":
        _load_openai_key(model)
    else:
        _load_hf_token(model)

    mode = _env("MODE")
    if not mode or mode.lower() not in {"white", "black"}:
        while True:
            mode = input("Select MODE ('white' or 'black'): ").strip().lower()
            if mode in {"white", "black"}:
                if provider == "openai" and mode == "white":
                    mode = "black"
                    print("OpenAI cannot run in white-box mode. Mode changed to black-box")
                os.environ["MODE"] = mode
                break
            print("Invalid input. Please type exactly 'white' or 'black'.")
    else:
        mode = mode.lower()

    if provider == "openai" and mode == "white":
        mode = "black"
        os.environ["MODE"] = mode
        print("OpenAI cannot run in white-box mode. Mode changed to black-box")

    return {"PROVIDER": provider, "MODEL": model, "MODE": mode}


def _load_hf_token(model: str) -> None:
    """Load HF token from .env or prompt; retry until valid and model-accessible."""
    from huggingface_hub import HfApi, model_info
    from huggingface_hub.errors import GatedRepoError, HfHubHTTPError

    while True:
        if not _env("HF_TOKEN"):
            while True:
                token = input("Paste your Hugging Face Token: ").strip()
                if token:
                    os.environ["HF_TOKEN"] = token
                    break
                print("Token cannot be empty. Please paste a valid Hugging Face token.")
        try:
            token = os.environ["HF_TOKEN"]
            user = HfApi().whoami(token=token)
            model_info(model, token=token)
            print(f"✓ Token valid — logged in as: {user['name']}.")
            print(f"✓ Access confirmed for: {model}")
            return
        except HfHubHTTPError:
            print("✗ Invalid or expired Hugging Face token.")
            os.environ.pop("HF_TOKEN", None)
        except GatedRepoError:
            print(
                f"✗ Token valid but access not granted for '{model}'.\n"
                f"  Request access at: https://huggingface.co/{model}"
            )
            return
        except Exception as e:
            print(f"✗ Could not verify Hugging Face token or model access: {e}")
            return


def _load_openai_key(model: str) -> None:
    """Load OpenAI key from .env or prompt; retry until valid and model-accessible."""
    from openai import AuthenticationError, OpenAI, PermissionDeniedError

    while True:
        if not _env("OPENAI_API_KEY"):
            while True:
                key = input("Paste your OpenAI API Key: ").strip()
                if key:
                    os.environ["OPENAI_API_KEY"] = key
                    break
                print("API key cannot be empty. Please paste a valid OpenAI API key.")
        try:
            client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
            client.models.retrieve(model)
            print(f"✓ OpenAI key valid — access confirmed for: {model}")
            return
        except AuthenticationError:
            print("✗ Invalid or expired OpenAI API key.")
            os.environ.pop("OPENAI_API_KEY", None)
        except PermissionDeniedError:
            print(
                f"✗ Key valid but no access to '{model}'.\n"
                f"  Check your plan at: https://platform.openai.com/account/limits"
            )
            return
        except Exception as e:
            print(f"✗ Could not verify OpenAI key or model access: {e}")
            return


config = set_config()

PROVIDER = config["PROVIDER"]
MODEL = config["MODEL"]
MODE = config["MODE"]

if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI

elif PROVIDER == "huggingface":
    from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint


if MODE == "white":
    from lm_polygraph.utils.model import WhiteboxModel
    from uqlm import WhiteBoxUQ, BlackBoxUQ, LongTextUQ
else:
    from lm_polygraph.utils.model import BlackboxModel
    from uqlm import BlackBoxUQ, LongTextUQ


print(f"\n✓ Setup completed — MODE: {MODE} | PROVIDER: {PROVIDER} | MODEL: {MODEL}")


##### Fix a random seed to ensure reproducibility of your experiment

In [ ]:
RANDOM_SEED = 42

def set_global_seed(seed=RANDOM_SEED):
    """Locks the random seed for predictable, reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed()


###Building the UQ Wrappers (Polygraph & UQLM)
Now we bring the architecture to life by constructing the specific **Wrappers** required by each library. These wrappers act as the crucial translation layer, converting raw LLM outputs into a format that the Uncertainty Quantification (UQ) algorithms can mathematically process.

**1. Wrapper A: The `lm_polygraph` Interfaces**
* **`WhiteboxModel`:** This wrapper directly ingests the raw, locally downloaded model weights (`base_model`) and its `tokenizer`. By wrapping the physical model, it exposes the deepest mathematical layers of the LLM—such as hidden states, attention matrices, and logits—directly to Polygraph's advanced estimators.
* **`BlackboxModel`:** Instead of local weights, this wrapper encapsulates an external API connection. While it typically treats the LLM as a pure text-in/text-out oracle, setting `supports_logprobs=True` gracefully upgrades it into a "Grey-box." This allows Polygraph to compute token-level entropy using API probabilities without ever needing the physical model.

**2. Wrapper B: The `uqlm` Interfaces (via LangChain)**
* **The LangChain Bridge:** Unlike Polygraph, `uqlm` does not interface with models directly. It requires a LangChain adapter (`ChatOpenAI` or `ChatHuggingFace`) to wrap the model, standardizing the connection regardless of the underlying provider.
* **`WhiteBoxUQ` Scorer:** In the `uqlm` dictionary, "White-box" means probability-based. This wrapper takes the LangChain object and leverages extracted `logprobs` from the API, enabling fast, single-generation mathematical scoring.
* **`BlackBoxUQ` Scorer:** This wrapper ignores probabilities entirely. It wraps the model to perform purely text-based sampling UQ, evaluating uncertainty based on the semantic consistency across multiple generated responses.

In [ ]:
from lm_polygraph.utils.generation_parameters import GenerationParameters

print(f" Setting up lm_polygraph in {MODE.upper()}-BOX mode...")
shared_generation_params = GenerationParameters()
shared_generation_params.temperature = 0.7
shared_generation_params.do_sample = True
shared_generation_params.max_new_tokens = 256

if MODE == "white":
    # Load heavy weights into GPU exactly once (16-bit precision)
    tokenizer = AutoTokenizer.from_pretrained(MODEL, token=os.environ["HF_TOKEN"])
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        device_map="auto",  # picks GPU if available, falls back to CPU
        torch_dtype=torch.float16,
        token=os.environ["HF_TOKEN"]
    )
    polygraph_model = WhiteboxModel(base_model, tokenizer, model_path=MODEL, generation_parameters=shared_generation_params)
else:
  # Pass only the credential for the active PROVIDER: lm-polygraph's BlackboxModel
  # prioritizes openai_api_key whenever it is not None, so passing both keys at once
  # (e.g. because OPENAI_API_KEY is also set for the claim-extraction pipeline) would
  # silently route HuggingFace models through the OpenAI client and fail.
  blackbox_kwargs = (
      {"openai_api_key": os.environ.get("OPENAI_API_KEY")} if PROVIDER == "openai"
      else {"hf_api_token": os.environ.get("HF_TOKEN")}
  )
  try:
        polygraph_model = BlackboxModel(
            model_path=MODEL,
            supports_logprobs=True,
            generation_parameters=shared_generation_params,
            **blackbox_kwargs,
        )
  except Exception as e:
        # Catching the exception
        print(f"WARNING: Failed to initialize BlackboxModel with logprobs. Error: {e}")
        print("Attempting graceful fallback to pure Black-box mode (supports_logprobs=False)...")

        # Fallback initialization without logprobs
        polygraph_model = BlackboxModel(
            model_path=MODEL,
            supports_logprobs=False,
            generation_parameters=shared_generation_params,
            **blackbox_kwargs,
        )

In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
import types
import asyncio
from langchain_core.messages import SystemMessage, HumanMessage


print(f"📊 Setting up uqlm in {MODE.upper()}-BOX mode...")

# Build the shared LangChain bridge
if PROVIDER == "openai":
    langchain_llm = ChatOpenAI(
        model=MODEL,
        api_key=os.environ["OPENAI_API_KEY"],
        model_kwargs={"logprobs": True} if MODE == "white" else {}
    )

elif PROVIDER == "huggingface":
    if MODE == "white":
        print("   ↳ Bridging the LOCAL GPU model directly to LangChain...")
        pipe = pipeline(
            "text-generation",
            model=base_model,
            tokenizer=tokenizer,
            max_new_tokens=256,
            return_full_text=False,
            do_sample=True,
            temperature=0.7
        )

        local_hf_llm = HuggingFacePipeline(pipeline=pipe)

        langchain_llm = ChatHuggingFace(llm=local_hf_llm)

    else:
        print("   ↳ Using Remote HuggingFace API (Warning: May be unstable for async UQ)...")
        hf_endpoint = HuggingFaceEndpoint(
            repo_id=MODEL,
            huggingfacehub_api_token=os.environ["HF_TOKEN"],
            task="text-generation",
            temperature=0.7,
            max_new_tokens=256,
            do_sample=True,
            return_full_text=False,
        )
        langchain_llm = ChatHuggingFace(llm=hf_endpoint)



print("  Applying async patch")

gpu_lock = asyncio.Lock()

async def _agenerate_shim(self, messages, stop=None, run_manager=None, **kwargs):
    sanitized_messages = []
    system_buffer = ""

    for msg in messages:
        if isinstance(msg, SystemMessage):
            system_buffer += msg.content + "\n\n"
        elif isinstance(msg, HumanMessage):
            if system_buffer:
                msg.content = system_buffer + msg.content
                system_buffer = ""
            sanitized_messages.append(msg)
        else:
            sanitized_messages.append(msg)
    # --------------------------------------------------------------------

    async with gpu_lock:
        return await asyncio.to_thread(
            self._generate,
            sanitized_messages,
            stop=stop,
            run_manager=run_manager,
            **kwargs
        )

langchain_llm._agenerate = types.MethodType(_agenerate_shim, langchain_llm)



### UQEngineContext

A centralized **Context Object** that manages the configuration and dependencies for the UQ framework.

* **State & Models:** Stores the execution modes (`"white"` or `"black"`) and holds the initialized model instances for both Polygraph and UQLM.
* **Auto-Validation:** Uses `__post_init__` to automatically enforce valid modes upon instantiation, ensuring a strict *fail-fast* architecture.

In [ ]:
from dataclasses import dataclass
from typing import Any, Optional

@dataclass
class UQEngineContext:
    mode: str                          # "white" or "black" — shared by both libraries
    polygraph_model: Optional[Any] = None
    langchain_llm: Optional[Any]   = None

    def __post_init__(self):
        if self.mode not in ["white", "black"]:
            raise ValueError("'mode' must be 'white' or 'black'.")

In [ ]:
uq_engine = UQEngineContext(
    mode=MODE,
    polygraph_model=polygraph_model,
    langchain_llm=langchain_llm,
)
print(f"UQ engine mode: {uq_engine.mode}")

### UQ_REGISTRY & Access Hierarchy

A centralized dictionary that configures and routes all supported Uncertainty Quantification (UQ) techniques across different libraries.

* **Hierarchical Access (`MODE_LEVELS`):** Assigns numerical privilege levels (`black: 0`, `white: 1`) to enforce strict authorization, ensuring models meet the minimum required access to run a specific technique.


In [ ]:
# --- Imports for lm_polygraph Execution ---
from lm_polygraph.estimators import *
from lm_polygraph import estimate_uncertainty

# ==========================================
# THE UQ REGISTRY (
# ==========================================

# Defining the hierarchy:
MODE_LEVELS = {
    "black": 0,
    "white": 1
}

UQ_REGISTRY = {
    # --- lm_polygraph techniques  ---
    "polygraph_attention": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence", "claim"],
        "min_required_mode": "white",
        "estimator_class": AttentionScore,
        "description": "Estimates uncertainty based on model’s attention weights."
    },
    "polygraph_max_token_prob": {
        "library": "lm_polygraph",
        "supported_granularity": ["token"],
        "min_required_mode": "black",
        "estimator_class": MaximumTokenProbability,
        "description": "Estimates token-level uncertainty by calculating log-probability."
    },

    # --- uqlm techniques ---
    "uqlm_exact_match": {
        "library": "uqlm",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "wrapper_class": BlackBoxUQ,
        "description": "Black-box technique computing exact match consistency."
    },
    "uqlm_entailment": {
        "library": "uqlm",
        "supported_granularity": ["sequence", "claim"],
        "min_required_mode": "black",
        "wrapper_class": {
            "sequence": BlackBoxUQ,
            "claim": LongTextUQ
        },
        "description": "Entailment Probability computes mean entailment via an NLI model."
    }
}

###  `show_help` (CLI Discovery Utility)

An internal helper function that enhances framework **discoverability** by rendering a clean, tabular summary of all registered UQ techniques directly in the terminal.

* **Dynamic Filtering:** Allows users to narrow down the available techniques by passing optional search parameters (`library`, `mode`, `granularity`).

In [ ]:
def show_help(library: str = None, mode: str = None, granularity: str = None):
    """
    Internal CLI/Help function.
    Prints a formatted summary table of all registered UQ techniques.

    Optional parameters to filter the output:
    - library (str): e.g., "uqlm" or "lm_polygraph"
    - mode (str): e.g., "white" or "black" (filters by minimum requirement)
    - granularity (str): e.g., "token", "sequence", "claim"
    """
    print("\n🔍 UQ FRAMEWORK - AVAILABLE TECHNIQUES")

    # --- 1. Table Structure Definition ---
    # We use f-string alignment modifiers (e.g., <25 means "left-align, occupy 25 characters")
    header = f"{'TECHNIQUE':<26} | {'LIBRARY':<13} | {'MODE':<6} | {'GRANULARITY':<18} | {'DESCRIPTION'}"
    separator = "-" * 120

    print(separator)
    print(header)
    print(separator)

    count = 0
    for tech_name, info in UQ_REGISTRY.items():
        # --- 2. Safe Extraction (Fail-Safe) ---
        lib = info.get("library", "N/A")
        req_mode = info.get("min_required_mode", "N/A")
        gran = ", ".join(info.get("supported_granularity", []))
        desc = info.get("description", "No description provided.")

        # --- 3. Filtering Engine ---
        if library and lib.lower() != library.lower():
            continue

        if mode and req_mode.lower() != mode.lower():
            continue

        if granularity and granularity.lower() not in [g.lower() for g in info.get("supported_granularity", [])]:
            continue

        # --- 4. Visual Cleanup ---
        # Truncate the description if it's too long to keep the table clean in the terminal
        max_desc_len = 200
        if len(desc) > max_desc_len:
            desc = desc[:max_desc_len - 3] + "..."

        # Print the formatted row in columns
        row = f"{tech_name:<26} | {lib:<13} | {req_mode:<6} | {gran:<18} | {desc}"
        print(row)
        count += 1

    print(separator)
    print(f"Showing {count} techniques based on applied filters.\n")

In [ ]:
show_help()

### The Core Execution Engine (Dispatcher & Handlers)

A state-of-the-art routing architecture that separates validation logic from library-specific execution using the **Facade + Handlers** pattern.

* **`evaluate_uncertainty` (The Dispatcher):** The universal public interface to **compute the uncertainty score**. It performs *fail-fast* registry validation, enforces hierarchical access controls (preventing Privilege Escalation between `white` and `black` modes), and securely routes the request to the correct underlying library to generate the standardized UQ payload.
* **`_handle_polygraph_execution`:** The private sub-engine for `lm_polygraph`. It handles its specific synchronous API, complex token string decoding, and multi-step claim extraction pipelines.
* **`_handle_uqlm_execution`:** The private asynchronous sub-engine for `uqlm`. It dynamically resolves the correct wrapper class based on the requested granularity and parses the library's nested output dictionaries.

In [ ]:
# ==========================================
#  AUXILIARY FUNCTIONS (Private logic)
# ==========================================
from lm_polygraph.stat_calculators import GreedyProbsCalculator, ClaimsExtractor
from lm_polygraph.utils.openai_chat import OpenAIChat

def _evaluate_claim_level_polygraph(prompt: str, estimator_class, model):
    """
    Handles the complex multi-step pipeline for Claim-level UQ in lm_polygraph.
    Requires OPENAI_API_KEY in the environment for the ClaimsExtractor.
    """
    print("   ↳ Initiating multi-step Claim-Level Pipeline...")

    if not _env("OPENAI_API_KEY"):
        print("\n   ⚠️ Warning: Missing OpenAI API key for ClaimsExtractor.")
        print("   Add OPENAI_API_KEY to your project .env file (see .env.example).")
        _load_openai_key("gpt-4")
        print("   OpenAI key set!\n")
    # ---------------------------------------------

    stat = {}
    texts = [prompt]

    print("   ↳ Step 1: Generating text and probabilities (GreedyProbsCalculator)...")
    greedy_calc = GreedyProbsCalculator()
    stat.update(greedy_calc(stat, texts, model))

    print("   ↳ Step 2: Extracting atomic claims (ClaimsExtractor via GPT-4)...")
    extractor = ClaimsExtractor(OpenAIChat("gpt-4"))
    stat.update(extractor(stat, texts, model))

    print(f"   ↳ Step 3: Applying {estimator_class.__name__}...")
    estimator = estimator_class()
    uncertainties = estimator(stat)


    print("   ↳ Step 4: Formatting the output...")
    claims_list = stat["claims"][0]
    scores_list = uncertainties[0]

    claim_details = []
    for claim_obj, score in zip(claims_list, scores_list):
        claim_details.append({
            "claim_text": claim_obj.claim_text,
            "score": float(score)
        })

    result_payload = {
        "input_prompt": prompt,
        "generated_text": stat["greedy_texts"][0],
        "uncertainty_score": claim_details
    }

    return result_payload

# ==========================================
# 2. PRIVATE HANDLERS (The Sub-Engines)
# ==========================================

def _handle_polygraph_execution(prompt: str, tech_info: dict, granularity: str, polygraph_model, **kwargs):
    """Handles all execution and parsing specifically for lm_polygraph."""
    print(f"⏳ Routing to Wrapper A (lm_polygraph) -> {tech_info['estimator_class'].__name__}...")

    # --- CLAIM ---
    if granularity == "claim":
        result_payload = _evaluate_claim_level_polygraph(prompt, tech_info["estimator_class"](**kwargs), polygraph_model)
        result_payload["library"] = "lm_polygraph"
        result_payload["estimator_name"] = tech_info["estimator_class"].__name__
        result_payload["granularity"] = granularity
        return result_payload

    # --- TOKEN ---
    elif granularity == "token":
        estimator = tech_info["estimator_class"](**kwargs)
        output = estimate_uncertainty(polygraph_model, estimator, input_text=prompt)
        import numpy as np

        raw_score = output.uncertainty
        if isinstance(raw_score, list) and len(raw_score) > 0 and isinstance(raw_score[0], np.ndarray):
            clean_score_list = raw_score[0].tolist()
        elif isinstance(raw_score, np.ndarray):
            clean_score_list = raw_score.tolist()
        else:
            clean_score_list = list(raw_score)

        raw_tokens = output.generation_tokens
        if len(raw_tokens) == 1 and isinstance(raw_tokens[0], list):
            raw_tokens = raw_tokens[0]

        if len(raw_tokens) > 0 and isinstance(raw_tokens[0], int):
            token_strings = polygraph_model.tokenizer.convert_ids_to_tokens(raw_tokens)
        else:
            token_strings = raw_tokens

        token_details = []
        min_len = min(len(token_strings), len(clean_score_list))
        for i in range(min_len):
            clean_token = str(token_strings[i]).replace("Ġ", " ").replace(" ", " ")
            token_details.append({
                "token": clean_token,
                "score": float(clean_score_list[i])
            })

        return {
            "library": "lm_polygraph",
            "estimator_name": output.estimator,
            "granularity": granularity,
            "input_prompt": output.input_text,
            "generated_text": output.generation_text,
            "uncertainty_score": token_details,
        }

    # --- SEQUENCE ---
    else:
        estimator = tech_info["estimator_class"](**kwargs)
        output = estimate_uncertainty(polygraph_model, estimator, input_text=prompt)
        import numpy as np

        if isinstance(output.uncertainty, np.ndarray) or isinstance(output.uncertainty, list):
            final_score = float(output.uncertainty[0])
        else:
            final_score = float(output.uncertainty)

        return {
            "library": "lm_polygraph",
            "estimator_name": output.estimator,
            "granularity": granularity,
            "input_prompt": output.input_text,
            "generated_text": output.generation_text,
            "uncertainty_score": final_score
        }

In [ ]:
async def _handle_uqlm_execution(prompt: str, technique_name: str, tech_info: dict, granularity: str, langchain_llm, **kwargs):
    """Handles all execution and parsing specifically for uqlm."""
    wrapper_map = tech_info["wrapper_class"]
    uqlm_class = wrapper_map[granularity] if isinstance(wrapper_map, dict) else wrapper_map

    # Split kwargs: constructor args (scorer_kwargs) vs. generate_and_score args (gen_kwargs)
    scorer_kwargs = {k: v for k, v in kwargs.items() if k not in ("num_responses", "sampling_temperature")}
    uqlm_wrapper = uqlm_class(llm=langchain_llm, scorers=[technique_name], **scorer_kwargs)

    gen_kwargs = {k: v for k, v in kwargs.items() if k in ("num_responses", "sampling_temperature")}
    uqlm_result = await uqlm_wrapper.generate_and_score(prompts=[prompt], **gen_kwargs)
    res_dict = uqlm_result.to_dict()

    # --- CLAIM ---
    if granularity == "claim":
        raw_claims_data = res_dict["data"]["claims_data"][0]
        claim_details = [{"claim_text": c["claim"], "score": 1.0 - float(c[technique_name])} for c in raw_claims_data]
        return {
            "library": "uqlm",
            "estimator_name": technique_name,
            "granularity": granularity,
            "input_prompt": prompt,
            "generated_text": res_dict["data"]["responses"],
            "uncertainty_score": claim_details
        }

    # --- TOKEN ---
    elif granularity == "token":
        raise ValueError("UQLM does not support token-level granularity.")

    # --- SEQUENCE ---
    else:
        confidence = res_dict["data"][technique_name][0]
        return {
            "library": "uqlm",
            "estimator_name": technique_name,
            "granularity": granularity,
            "input_prompt": prompt,
            "generated_text": res_dict["data"]["responses"],
            "uncertainty_score": 1.0 - float(confidence),
            "confidence_score": float(confidence),
        }


In [ ]:
async def evaluate_uncertainty(prompt: str, technique_name: str, library: str, granularity: str,
                         uq_context: UQEngineContext, **kwargs):
    """
    Universal interface for UQ evaluation. Routes to simple functions or
    complex pipelines based on the requested granularity, standardizing the output.
    """
    print(f"\n🧠 Processing Request: Library='{library}' | Technique='{technique_name}' | Granularity='{granularity}'")

    registry_key = f"{library}_{technique_name}"

    # --- Step A: Registry Validation ---
    if registry_key not in UQ_REGISTRY:
        raise ValueError(f"Technique combination '{registry_key}' is not in the UQ_REGISTRY. Please add it first.")

    tech_info = UQ_REGISTRY[registry_key]

    if granularity not in tech_info["supported_granularity"]:
        raise ValueError(
            f" Granularity Mismatch: '{technique_name}' in {library} only supports {tech_info['supported_granularity']}. "
            f"You requested '{granularity}'."
        )

    # UQEngineContext tracks a single mode shared by both libraries (see cell 5).
    current_mode = uq_context.mode
    min_required_mode = tech_info.get("min_required_mode", "black")

    # Translate mode strings into their corresponding numeric levels
    current_level = MODE_LEVELS.get(current_mode, -1)
    required_level = MODE_LEVELS.get(min_required_mode, 99)

    # If the current level is lower than the required one, block execution!
    if current_level < required_level:
        raise ValueError(
            f"Privilege Escalation Error: The technique '{registry_key}' requires "
            f"'{min_required_mode.upper()}' level access, but the notebook is running at "
            f"a lower level ('{current_mode.upper()}')."
        )
    # ----------------------------------------------

    print(f"Validation Passed. Library to use: {tech_info['library'].upper()} (Mode: {current_mode})")

    merged_kwargs = {**tech_info.get("default_kwargs", {}), **kwargs}

    # --- Step B: Execution Routing ---
    if tech_info["library"] == "lm_polygraph":
        if uq_context.polygraph_model is None:
            raise ValueError("'polygraph_model' is required by this technique but was not found in the UQEngineContext.")

        result_payload = _handle_polygraph_execution(prompt, tech_info, granularity, uq_context.polygraph_model, **merged_kwargs)

    elif tech_info["library"] == "uqlm":
        if uq_context.langchain_llm is None:
              raise ValueError(" 'langchain_llm' bridge is required by this technique but was not found in the UQEngineContext.")

        result_payload = await _handle_uqlm_execution(prompt, technique_name, tech_info, granularity, uq_context.langchain_llm, **merged_kwargs)

    print(f"🎯 {granularity.capitalize()}-level execution complete!")
    return result_payload


In [ ]:
test_result = await evaluate_uncertainty(
    prompt="What are the early signs of Parkinson's disease?",
    library="uqlm",
    technique_name="entailment",
    granularity="claim",
    uq_context=uq_engine
)
print(test_result)


# Text-Only

## Black Box Techniques

### Black-Box Section Setup

Run the cells below **before** the demonstrations.

| Cell | Purpose |
|---|---|
| `bb-setup-pip` | Installs `matplotlib`/`scipy` (needed for plots and Spearman correlation) |
| `bb-setup-imports` | Imports the black-box estimator classes used in this section |

> **Tip:** Set `MODE=black` in your `.env` file to run all black-box demos without downloading local model weights.
> This requires your Hugging Face account to have Inference Providers access for the chosen `MODEL` — not every gated/small model is served there. `MODE=white` (local weights) is the most reliably tested path in this notebook.


In [ ]:
%pip install -q matplotlib scipy


In [ ]:
# ── Black-box-specific estimator imports ──────────────────────
from lm_polygraph.estimators import (
    # Verbalized
    Verbalized1S, Verbalized2S, Linguistic1S,
    # P(True) — verbalized self-evaluation
    PTrue, PTrueClaim,
    # Consistency — graph-free
    LexicalSimilarity, SemanticEntropy,
    NumSemSets, LabelProb,
    # Consistency — graph-based
    DegMat, Eccentricity, EigValLaplacian,
    # Consistency — SAR (Sentence-level Answer Relevance)
    SAR, SentenceSAR,
    # Consistency — Semantic Density
    SemanticDensity,
    # Claim-level
    FrequencyScoringClaim,
    # Information-theoretic (grey-box: needs only per-token logprobs)
    MaximumSequenceProbability, Perplexity,
)
from scipy.stats import spearmanr
print("Black-box estimator classes loaded.")

---
# Black-Box Uncertainty Quantification

Black-box UQ treats the LLM as an **opaque text API** — no logits, no hidden states, no attention weights.
This is the default deployment setting when querying GPT-4, Gemini, or any closed commercial model.

## Why It Matters in Clinical AI

In hospital decision-support systems, you typically integrate a third-party LLM via API.
Model internals are inaccessible. Black-box UQ is therefore the **most practically relevant** approach for clinical deployment: you can always estimate uncertainty regardless of whether you own or can access the model weights.

## Taxonomy (AAAI-2026 / TACL)

| Family | Core idea | Inference cost | Calibration on small LLMs |
|---|---|---|---|
| **Verbalized** | Ask the model *"how confident are you?"* | 1–2 calls | ⚠️ Unreliable below ~70B params |
| **Consistency** | Sample K answers; measure semantic agreement | K calls | ✅ Model-agnostic — works at any scale |

## Method Selection: Decision Tree

From the AAAI-2026 tutorial visual guide:

```
Black-box model?
  YES
  ├── Have compute budget?
  │     YES → Consistency-based: EigLaplacian, DegMat, Eccentricity, SemanticEntropy
  │     NO  → Verbalized uncertainty
```

**EigLaplacian is the recommended consistency method** from the decision tree — it is the primary technique covered in Section 2.3 below.

## Granularity in Black-Box Mode

| Granularity | Black-box? | Notes |
|---|---|---|
| **Sequence** | ✅ Both libraries | One score per full response |
| **Claim** | ✅ UQLM (`LongTextUQ`) | Per-atomic-claim scores |
| **Token** | ❌ Not supported | Requires logits (white-box only) |

> 💡 **Convention:** UQLM returns **confidence** ∈ [0, 1]; we convert to **uncertainty** as `1 − confidence`
> so that all visualizations share the same direction: **higher = more uncertain**.

### Clinical Prompts

Reused across all black-box demonstrations.

In [ ]:
CLINICAL_PROMPTS = [
    "What is the normal resting heart rate for a healthy adult?",
    "What is the first-line treatment for type 2 diabetes?",
    "What are the early signs of Parkinson's disease?",
    "What is the recommended dose of aspirin for cardiovascular prevention?",
    "What is the most effective treatment for long COVID neurological symptoms?",
    "Describe the mechanism of action of tirzepatide on GIP and GLP-1 receptors.",
]
DEMO_PROMPT = CLINICAL_PROMPTS[2]
FACTUAL_PROMPT = CLINICAL_PROMPTS[0]
UNCERTAIN_PROMPT = CLINICAL_PROMPTS[4]
short_prompts = [p[:42] + "..." for p in CLINICAL_PROMPTS]
print(f"Demo prompt: {DEMO_PROMPT}")

### Visualization Utilities

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

CMAP_UNCERTAINTY = plt.cm.RdYlGn_r

def plot_sequence_comparison(results, labels, title="Sequence-Level Uncertainty"):
    arr = np.array(results, dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    colors = [CMAP_UNCERTAINTY(v) for v in norm]
    fig, ax = plt.subplots(figsize=(12, 4))
    bars = ax.bar(range(len(labels)), arr, color=colors, edgecolor="white", width=0.6)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("Uncertainty (↑ = less confident)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    for bar, val in zip(bars, arr):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f"{val:.3f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout(); plt.show()

def plot_claim_uncertainty(result, title=""):
    scores = result["uncertainty_score"]
    claims = [s["claim_text"][:55] + ("..." if len(s["claim_text"]) > 55 else "") for s in scores]
    values = [s["score"] for s in scores]
    arr = np.array(values)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    colors = [CMAP_UNCERTAINTY(v) for v in norm]
    fig, ax = plt.subplots(figsize=(14, max(4, len(claims) * 0.45)))
    ax.barh(range(len(claims)), arr, color=colors, edgecolor="white", height=0.7)
    ax.set_yticks(range(len(claims)))
    ax.set_yticklabels(claims, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Uncertainty")
    ax.set_title(title or "Claim-Level Uncertainty", fontweight="bold")
    plt.tight_layout(); plt.show()

print("Visualization utilities loaded.")

### Extending `UQ_REGISTRY` with Black-Box Techniques

Registry key pattern: `f"{library}_{technique_name}"` → e.g. `evaluate_uncertainty(..., library="polygraph", technique_name="verbalized_1s")` maps to `"polygraph_verbalized_1s"`.

In [ ]:
BB_REGISTRY_ENTRIES = {
    # ── VERBALIZED (lm-polygraph) ────────────────────────────────────
    "polygraph_verbalized_1s": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": Verbalized1S,
        "default_kwargs": {"confidence_regex": r"Probability:\s*(\d+\.?\d*)", "name_postfix": "_top1"},
        "description": "Verbalized 1S: answer + numeric confidence in one generation (Tian et al., 2023)."
    },
    "polygraph_verbalized_2s": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": Verbalized2S,
        "default_kwargs": {"confidence_regex": r"Probability:\s*(\d+\.?\d*)", "name_postfix": "_top1"},
        "description": "Verbalized 2S: separate generation then confidence-estimation turns."
    },
    "polygraph_linguistic_1s": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": Linguistic1S,
        "description": "Linguistic 1S: maps hedging phrases ('I think', 'maybe') to a confidence score."
    },
    # ── CONSISTENCY — graph-free (lm-polygraph) ──────────────────────
    "polygraph_lexical_similarity": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": LexicalSimilarity,
        "default_kwargs": {"metric": "rougeL"},
        "description": "Mean ROUGE-L overlap across K sampled answers. Cheap but surface-level."
    },
    "polygraph_semantic_entropy": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": SemanticEntropy,
        "default_kwargs": {"class_probability_estimation": "frequency"},
        "description": "Cluster K answers by NLI equivalence, compute Shannon entropy over clusters (Kuhn et al., 2023)."
    },
    "polygraph_num_sem_sets": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": NumSemSets,
        "description": "Number of distinct semantic equivalence classes — simpler SE proxy."
    },
    "polygraph_label_prob": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": LabelProb,
        "description": "Frequency of the greedy answer among K samples — black-box MSP proxy."
    },
    # ── CONSISTENCY — graph-based (lm-polygraph) ─────────────────────
    "polygraph_degmat": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": DegMat,
        "default_kwargs": {"similarity_score": "NLI_score", "affinity": "entail"},
        "description": "Degree Matrix: mean node degree of the NLI semantic similarity graph (Lin et al., 2023)."
    },
    "polygraph_eccentricity": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": Eccentricity,
        "default_kwargs": {"similarity_score": "NLI_score", "affinity": "entail"},
        "description": "Eccentricity: longest shortest path in the semantic similarity graph (Lin et al., 2023)."
    },
    "polygraph_eig_val_laplacian": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": EigValLaplacian,
        "default_kwargs": {"similarity_score": "NLI_score", "affinity": "entail"},
        "description": "EigLaplacian: largest eigenvalue of graph Laplacian — recommended method from AAAI-2026 decision tree."
    },
    # ── CONSISTENCY (uqlm) ───────────────────────────────────────────
    "uqlm_semantic_negentropy": {
        "library": "uqlm",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "wrapper_class": BlackBoxUQ,
        "description": "Semantic negentropy — normalized entropy over NLI-equivalence clusters."
    },
    "uqlm_cosine_sim": {
        "library": "uqlm",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "wrapper_class": BlackBoxUQ,
        "description": "Mean embedding cosine similarity between original and sampled responses."
    },
    "uqlm_noncontradiction": {
        "library": "uqlm",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "wrapper_class": BlackBoxUQ,
        "description": "Non-contradiction probability via NLI model across K samples."
    },
    # ── P(True) — verbalized self-evaluation (Kadavath et al., 2022) ──────
    "polygraph_p_true": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": PTrue,
        "description": "P(True): sample K answers then ask the model is this correct? — aggregates TRUE probability.",
    },
    "polygraph_p_true_claim": {
        "library": "lm_polygraph",
        "supported_granularity": ["claim"],
        "min_required_mode": "black",
        "estimator_class": PTrueClaim,
        "description": "P(True) at claim level: each extracted claim independently self-evaluated by the model.",
    },
    # ── SAR — Sentence-level Answer Relevance ────────────────────────
    "polygraph_sar": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": SAR,
        "description": "SAR: combined token+sentence relevance between question and K sampled answers (Kuhn et al., 2023).",
    },
    "polygraph_sentence_sar": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": SentenceSAR,
        "description": "SentenceSAR: sentence-level average relevance between question and sampled answers.",
    },
    # ── Semantic Density ────────────────────────────────────────────
    "polygraph_semantic_density": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": SemanticDensity,
        "description": "SemanticDensity: density of K sampled answers in embedding space — low density = high uncertainty.",
    },
    # ── Information-Theoretic — grey-box (needs only per-token logprobs) ───
    "polygraph_msp": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": MaximumSequenceProbability,
        "description": "MSP: negative sum of log-probabilities of generated tokens. Grey-box (needs per-token logprobs only).",
    },
    "polygraph_perplexity": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "estimator_class": Perplexity,
        "description": "Perplexity: length-normalised exponential of mean negative log-likelihood. Grey-box.",
    },
    # ── Claim-level: Frequency Scoring ────────────────────────────
    "polygraph_frequency_scoring_claim": {
        "library": "lm_polygraph",
        "supported_granularity": ["claim"],
        "min_required_mode": "black",
        "estimator_class": FrequencyScoringClaim,
        "description": "FrequencyScoringClaim: fraction of K samples that NLI-confirm each extracted claim. Pure black-box.",
    },
}

UQ_REGISTRY.update(BB_REGISTRY_ENTRIES)
print(f"Registry now contains {len(UQ_REGISTRY)} black-box techniques.")
show_help(mode="black")

---
## 1. Verbalized Uncertainty

The model is **explicitly prompted** to express confidence — no sampling required.

### How Each Variant Works

| Method | Mechanism | lm-polygraph class |
|---|---|---|
| **Verbalized 1S** | Answer + confidence number in a single generation | `Verbalized1S` |
| **Verbalized 2S** | First generate the answer; then ask a *separate* follow-up question about confidence | `Verbalized2S` |
| **Linguistic 1S** | Parse hedging language ("I think", "possibly", "it is likely") and map phrases to a numeric confidence | `Linguistic1S` |

### Mathematical Formulation (Verbalized 1S / 2S)

The model is prompted to produce:
```
Guess: <answer>
Probability: <p>
```
The extracted probability $\hat{p} \in [0,1]$ is then converted to uncertainty:
$$U_{\text{verb}} = 1 - \hat{p}$$

### Why Verbalization Fails on Small LLMs

Small models (< 7B parameters) struggle because:
1. **Instruction-following gap** — they often ignore the exact `Probability: X.XX` format, producing free-form text the regex cannot parse.
2. **Miscalibration** — they tend to output `Probability: 1.0` (overconfident) regardless of actual uncertainty.
3. **Sycophancy** — they mimic the expected "confident expert" persona even when they should hedge.

> 💡 UQLM has no verbalized scorer. Its black-box core relies entirely on consistency — which is why UQLM typically outperforms lm-polygraph in pure black-box mode on small models.

In [ ]:
VERBALIZED_TEMPLATE = (
    "You are a clinical assistant. Answer the medical question concisely.\n"
    "Then state your confidence as a probability between 0.0 and 1.0.\n"
    "Format exactly:\nGuess: <your answer>\nProbability: <number>\n\n"
    "Question: {question}"
)

verb_prompt = VERBALIZED_TEMPLATE.format(question=DEMO_PROMPT)
print(verb_prompt)

In [ ]:
print("Running Verbalized 1S (lm-polygraph)...")
v1s = await evaluate_uncertainty(
    prompt=verb_prompt,
    library="polygraph", technique_name="verbalized_1s", granularity="sequence",
    uq_context=uq_engine
)
print(f"Uncertainty: {v1s['uncertainty_score']:.4f}")
print(f"Generated:\n{v1s['generated_text'][:400]}...")

In [ ]:
print("Running Verbalized 2S (lm-polygraph)...")
v2s = await evaluate_uncertainty(
    prompt=verb_prompt,
    library="polygraph", technique_name="verbalized_2s", granularity="sequence",
    uq_context=uq_engine
)
print(f"Uncertainty: {v2s['uncertainty_score']:.4f}")

In [ ]:
print("Running Linguistic 1S (lm-polygraph)...")
ling_prompt = f"As a medical expert, answer this question. You may use hedging if unsure.\n\nQuestion: {DEMO_PROMPT}"
ling = await evaluate_uncertainty(
    prompt=ling_prompt,
    library="polygraph", technique_name="linguistic_1s", granularity="sequence",
    uq_context=uq_engine
)
print(f"Uncertainty: {ling['uncertainty_score']:.4f}")

In [ ]:
# ── Verbalized comparison across all clinical prompts ───────────────
print("Running all three verbalized methods across all clinical prompts...\n")

verb_scores   = {"verbalized_1s": [], "verbalized_2s": [], "linguistic_1s": []}

for prompt in CLINICAL_PROMPTS:
    vp = VERBALIZED_TEMPLATE.format(question=prompt)
    lp = f"As a medical expert, answer this question. You may use hedging if unsure.\n\nQuestion: {prompt}"

    r1s = await evaluate_uncertainty(vp,  "polygraph", "verbalized_1s",  "sequence", uq_engine)
    r2s = await evaluate_uncertainty(vp,  "polygraph", "verbalized_2s",  "sequence", uq_engine)
    rl  = await evaluate_uncertainty(lp,  "polygraph", "linguistic_1s",  "sequence", uq_engine)

    verb_scores["verbalized_1s"].append(r1s["uncertainty_score"])
    verb_scores["verbalized_2s"].append(r2s["uncertainty_score"])
    verb_scores["linguistic_1s"].append(rl["uncertainty_score"])
    print(f"P{CLINICAL_PROMPTS.index(prompt)+1}: 1S={r1s['uncertainty_score']:.3f} | 2S={r2s['uncertainty_score']:.3f} | Ling={rl['uncertainty_score']:.3f}")

# ── Side-by-side bar chart ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=False)
method_labels = ["Verbalized 1S", "Verbalized 2S", "Linguistic 1S"]
keys          = ["verbalized_1s", "verbalized_2s", "linguistic_1s"]

for ax, key, title in zip(axes, keys, method_labels):
    arr  = np.array(verb_scores[key], dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Uncertainty")
    for j, v in enumerate(arr):
        ax.text(j, v + arr.max()*0.02, f"{v:.2f}", ha="center", fontsize=7)

plt.suptitle("Verbalized Methods — All Clinical Prompts", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

> **Observation — Verbalized Methods:**
>
> All three verbalized methods tend to produce **flat, undifferentiated scores** when run on small models
> (< 7B parameters such as Gemma-2-2B).
> The model typically outputs `Probability: 1.0` (full confidence) for *every* clinical prompt —
> including P5 (long COVID neurological symptoms) and P6 (tirzepatide mechanism), which are genuinely
> uncertain because they concern recent research not well represented in training data.
>
> **1S vs 2S:** The two-step variant (2S) is slightly better calibrated in theory — separating "generate"
> from "evaluate" reduces the pressure on a single generation. In practice on small models, neither
> reliably produces granular confidence values.
>
> **Linguistic 1S** is the most robust of the three on small models because it does not require the model
> to produce a number at all — it simply checks whether the generated text *already* contains hedging
> language. When the model does hedge (e.g., "it is thought that...", "research suggests..."),
> Linguistic 1S correctly elevates uncertainty.
>
> **Takeaway:** For clinical deployment with small open-source models, verbalized uncertainty should be
> treated as a weak signal. Prefer consistency-based methods (Section 2) for reliable uncertainty estimates.

---
### 1.4 P(True) — Self-Evaluation Verbalized Method

**P(True)** (Kadavath et al., 2022) is a distinct verbalized method that uses a **two-stage self-evaluation pipeline** instead of asking the model to output a confidence number:

1. **Generate** an answer to the question
2. **Evaluate**: prompt the model with question + its own answer and ask:
   *"Is the proposed answer: (A) True or (B) False?"*

The fraction of "True" responses across K evaluations gives the confidence:

$$P(\text{True}) = \frac{\#\text{True responses}}{K}, \quad U_{\text{P(True)}} = 1 - P(\text{True})$$

**Why P(True) Is More Robust Than Verbalized 1S/2S:**

| Method | Format required | When estimated |
|---|---|---|
| **Verbalized 1S** | `Probability: X.XX` (exact decimal) | During generation |
| **Verbalized 2S** | `Probability: X.XX` (exact decimal) | Follow-up turn |
| **P(True)** | `(A) True` or `(B) False` (multiple choice) | After full answer |

Multiple-choice prompting is far more reliable than free-form decimal prediction, especially for small models trained via RLHF — they are optimized for option-selection tasks.

**Claim-Level Extension — `PTrueClaim`:**
`PTrueClaim` applies P(True) to each **atomic claim** extracted from the response independently, giving a per-claim reliability profile directly usable for clinical safety review.

> **Note:** `PTrueClaim` uses the `ClaimsExtractor` pipeline (requires OpenAI API key for GPT-4 claim decomposition) — covered in Section 3.

In [ ]:
# ── P(True): self-evaluation across all clinical prompts ────────────────
print("=" * 60)
print("P(True) — self-evaluation verbalized method (Kadavath et al., 2022)")
print("=" * 60)

ptrue_scores = []

for prompt in CLINICAL_PROMPTS:
    result_seq = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="p_true",
        granularity="sequence", uq_context=uq_engine
    )
    ptrue_scores.append(result_seq["uncertainty_score"])
    print(f"  [U={result_seq['uncertainty_score']:.4f}] {prompt[:55]}")

plot_sequence_comparison(ptrue_scores, short_prompts, "P(True) — Self-Evaluation Uncertainty (Sequence Level)")

# ── Compare P(True) vs Verbalized 1S ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, (scores, title) in zip(axes, [
    (verb_scores["verbalized_1s"], "Verbalized 1S\n(decimal probability format)"),
    (ptrue_scores,                 "P(True)\n(self-evaluation A/B format)"),
]):
    arr  = np.array(scores, dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Uncertainty")
    for j, v in enumerate(arr):
        ax.text(j, v + max(arr) * 0.02, f"{v:.2f}", ha="center", fontsize=7)

plt.suptitle("Verbalized 1S vs P(True) — Clinical Prompts", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

> **Observation — P(True):**
>
> P(True) typically shows **more variation** across prompts than Verbalized 1S/2S because the A/B
> multiple-choice format is easier for small models to follow than generating a precise decimal.
>
> - **P1 (heart rate)**: High P(True) → low uncertainty — the model consistently self-assesses as correct
> - **P5/P6 (long COVID, tirzepatide)**: Lower P(True) → higher uncertainty — self-assessments diverge
>
> **Key difference from 1S/2S:** Verbalized 1S/2S often outputs `Probability: 1.0` for every prompt
> (regex failure or overconfidence). P(True) aggregates K binary judgments — even a slightly uncertain
> model will sometimes output "False", yielding a non-trivial uncertainty score.
>
> **Best use case:** Use P(True) as a complementary verbalized signal alongside consistency methods.
> For very small models (< 3B), both verbalized methods degrade; prefer consistency-based UQ as primary signal.

---
### 1.5 Information-Theoretic Methods (Grey-Box)

**MSP** (Maximum Sequence Probability) and **Perplexity** need only the log-probability of the
*tokens the model actually generated* — no full vocabulary distribution required. This makes them
achievable in **grey-box** mode: an API that returns per-token logprobs (this notebook's
`BlackboxModel(supports_logprobs=True)`) is enough — no local weights needed.

$$U_{\text{MSP}} = -\sum_{t} \log P(x_t \mid x_{<t}), \qquad U_{\text{PPL}} = \exp\left(\frac{1}{T}\sum_t -\log P(x_t \mid x_{<t})\right)$$

Both are single-generation baselines: no sampling, no NLI model — the cheapest sequence-level
signal in this notebook whenever your API exposes logprobs.


In [ ]:
# ── MSP + Perplexity: sequence log-likelihood grey-box methods ──────────────
print("=" * 60)
print("MSP & Perplexity — sequence log-likelihood uncertainty (grey-box)")
print("=" * 60)

msp_scores, ppl_scores = [], []
for prompt in CLINICAL_PROMPTS:
    r_msp = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="msp",
        granularity="sequence", uq_context=uq_engine
    )
    r_ppl = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="perplexity",
        granularity="sequence", uq_context=uq_engine
    )
    msp_scores.append(r_msp["uncertainty_score"])
    ppl_scores.append(r_ppl["uncertainty_score"])
    print(f"  MSP={r_msp['uncertainty_score']:.4f} | PPL={r_ppl['uncertainty_score']:.4f} | {prompt[:48]}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, (scores, title) in zip(axes, [(msp_scores, "MSP (neg. log seq. prob.)"), (ppl_scores, "Perplexity")]):
    arr  = np.array(scores, dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Uncertainty")
plt.suptitle("MSP & Perplexity — Clinical Prompts", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


> **Observation — MSP & Perplexity:**
>
> Both track each other closely (Perplexity is just the length-normalised exponential of the same
> log-likelihood sum MSP leaves raw), and both agree with the consistency methods in Section 2 on
> the coarse ranking — but neither needs K samples, making them the cheapest sequence-level signal
> here whenever the API exposes logprobs.
>
> **Caution:** MSP is length-sensitive — a longer correct answer can score worse than a short wrong
> one purely because more token probabilities are multiplied together. Perplexity's length
> normalisation mitigates this; prefer consistency methods when answer lengths vary a lot.


---
## 2. Consistency-Based Methods (Sequence Level)

Sample **K stochastic answers** to the same prompt and measure how much they agree.
Higher disagreement → higher uncertainty. Model-agnostic and generally more reliable than verbalization at any model size.

### 2.1 Semantic Entropy (Kuhn et al., 2023)

The key insight is that surface-level diversity (different words) is noise; **semantic** diversity (different meanings) is the real signal.

**Algorithm:**
1. Sample K answers: $\{a_1, ..., a_K\}$
2. Build equivalence classes $C = \{c_1, ..., c_M\}$ where $a_i \sim a_j$ iff an NLI model scores mutual entailment
3. Estimate class probabilities: $p(c) = |c| / K$
4. Compute Shannon entropy: $SE = -\sum_{c} p(c) \log p(c)$

$$U_{\text{SE}} = -\sum_{c \in C} \frac{|c|}{K} \log \frac{|c|}{K}$$

If all K answers fall into one class → $SE = 0$ (certain). If every answer is unique → $SE = \log K$ (maximally uncertain).

### 2.2 Graph-Based Methods (Lin et al., 2023)

All graph-based methods share the same **semantic similarity graph** construction:

- **Nodes**: the K sampled answers $\{a_1, ..., a_K\}$
- **Edges**: weighted by NLI entailment score $W_{ij} = \text{NLI}(a_i \to a_j) \cdot \text{NLI}(a_j \to a_i)$

From this graph, different spectral/topological features capture uncertainty:

| Method | Formula | Intuition |
|---|---|---|
| **Degree Matrix (DegMat)** | $U = -\frac{1}{K}\sum_i \sum_j W_{ij}$ | Low total weight → answers are semantically distant → uncertain |
| **Eccentricity** | $U = \max_i \min_j \, d(a_i, a_j)$ | Maximum shortest-path distance — how isolated is the most peripheral answer |
| **EigLaplacian** | $U = \lambda_1(\mathcal{L})$, where $\mathcal{L} = D - W$ | Largest eigenvalue of the graph Laplacian — captures cluster separation |

### 2.3 EigLaplacian — the Recommended Method

The **Laplacian spectrum** is the recommended consistency technique in the AAAI-2026 decision tree.
The Laplacian matrix $\mathcal{L} = D - W$ encodes the full connectivity structure of the semantic graph.
Its **largest eigenvalue** $\lambda_1$ measures how "spread out" the answers are:
- $\lambda_1 \approx 0$ → all answers form one tight semantic cluster → low uncertainty
- $\lambda_1 \gg 0$ → answers form multiple disconnected clusters → high uncertainty

This is more robust than DegMat and Eccentricity because it captures **global graph structure** rather than local node properties.

### 2.4 Lexical Similarity (fast baseline)

$$U_{\text{lex}} = 1 - \frac{1}{K(K-1)} \sum_{i \ne j} \text{ROUGE-L}(a_i, a_j)$$

Cheapest consistency method — no NLI model required. Fails when answers paraphrase each other (same meaning, different wording).

In [ ]:
NUM_SAMPLES = 5  # increase to 10 for paper-faithful benchmarks (slower)
polygraph_model.generation_parameters.temperature = 0.7
polygraph_model.generation_parameters.do_sample = True
print(f"Using K={NUM_SAMPLES} samples for consistency methods.")

### 2.5 lm-polygraph Consistency Estimators

Running all lm-polygraph black-box techniques across the 6 clinical prompts.
EigLaplacian is run first as the highlighted method from the decision tree.

In [ ]:
# ── EigLaplacian: run first as the decision-tree recommended method ──
print("=" * 60)
print("EigLaplacian — recommended consistency method (AAAI-2026)")
print("=" * 60)
eig_scores = []
for prompt in CLINICAL_PROMPTS:
    result = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="eig_val_laplacian",
        granularity="sequence", uq_context=uq_engine
    )
    eig_scores.append(result["uncertainty_score"])
    print(f"  λ₁={result['uncertainty_score']:.4f}  |  {prompt[:55]}")

plot_sequence_comparison(eig_scores, short_prompts, "EigLaplacian (λ₁ of Graph Laplacian) — Clinical Prompts")

# ── All other lm-polygraph techniques ───────────────────────────────
POLY_BB_TECHNIQUES = [
    ("lexical_similarity", "Lexical Similarity (ROUGE-L)"),
    ("semantic_entropy",   "Semantic Entropy"),
    ("degmat",             "Degree Matrix"),
    ("eccentricity",       "Eccentricity"),
    ("num_sem_sets",       "Num Semantic Sets"),
    ("label_prob",         "Label Probability"),
]

poly_bb_scores = {"eig_val_laplacian": eig_scores}
for technique, label in POLY_BB_TECHNIQUES:
    print(f"\n▶ {label}")
    scores = []
    for prompt in CLINICAL_PROMPTS:
        result = await evaluate_uncertainty(
            prompt=prompt, library="polygraph", technique_name=technique,
            granularity="sequence", uq_context=uq_engine
        )
        scores.append(result["uncertainty_score"])
        print(f"  [{result['uncertainty_score']:.4f}] {prompt[:50]}")
    poly_bb_scores[technique] = scores

# ── Individual plots for key methods ────────────────────────────────
for key, title in [
    ("semantic_entropy",   "Semantic Entropy"),
    ("degmat",             "Degree Matrix"),
    ("lexical_similarity", "Lexical Similarity (ROUGE-L)"),
]:
    plot_sequence_comparison(poly_bb_scores[key], short_prompts, f"{title} — Clinical Prompts")

In [ ]:
# ── Spearman correlation between lm-polygraph consistency methods ───
from scipy.stats import spearmanr

poly_methods = ["eig_val_laplacian", "semantic_entropy", "degmat", "eccentricity",
                "lexical_similarity", "num_sem_sets", "label_prob"]
labels_short  = ["EigLap", "SemEnt", "DegMat", "Ecc", "LexSim", "NumSem", "LabelP"]

n = len(poly_methods)
corr_matrix = np.zeros((n, n))
for i, m1 in enumerate(poly_methods):
    for j, m2 in enumerate(poly_methods):
        rho, _ = spearmanr(poly_bb_scores[m1], poly_bb_scores[m2])
        corr_matrix[i, j] = rho

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(labels_short, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(labels_short)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{corr_matrix[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr_matrix[i,j]) > 0.6 else "black")
plt.colorbar(im, ax=ax, label="Spearman ρ")
ax.set_title("Rank Correlation — lm-polygraph Black-Box Methods", fontweight="bold")
plt.tight_layout()
plt.show()

> **Observation — lm-polygraph Consistency Methods:**
>
> **EigLaplacian** correctly identifies P5 (long COVID neurological) and P6 (tirzepatide mechanism)
> as the highest-uncertainty prompts — these cover active research areas where the model generates
> semantically diverse answers across samples, reflecting genuine knowledge gaps.
> P1 (resting heart rate) receives the lowest $\lambda_1$, confirming that the model's K samples
> all cluster tightly around "60–100 bpm."
>
> **DegMat ↔ EigLaplacian (ρ ≈ high):** Both operate on the same NLI graph and capture global
> connectivity — they rank prompts almost identically. EigLaplacian is preferred because it is
> more robust to outlier samples (eigenvalue decomposition is less sensitive to individual edge weights).
>
> **Semantic Entropy ↔ NumSemSets (ρ ≈ high):** NumSemSets counts equivalence classes while SE
> measures their entropy — strongly correlated but SE is more expressive (it weighs class sizes).
>
> **LexicalSimilarity (ρ ≈ low vs. NLI methods):** Surface ROUGE-L diverges from NLI-based methods
> on medical prompts because the model often paraphrases the same correct answer in different wording
> (e.g., "60–100 beats per minute" vs "60 to 100 bpm"). NLI correctly identifies these as identical;
> ROUGE penalises the wording difference as if they were distinct answers.
>
> **Clinical implication:** For medical QA, always prefer NLI-based methods (SE, DegMat, EigLap)
> over surface-level ROUGE. The difference is especially large on P3 (Parkinson's signs) where
> the model produces long, varied phrasings of the same clinical facts.

### 2.6 UQLM Consistency Scorers

UQLM exposes four black-box scorers, all built around semantic consistency but using different comparison mechanisms:

| Scorer | Comparison signal | Notes |
|---|---|---|
| `exact_match` | String equality | Fastest; good for MCQ / short factual answers |
| `entailment` | NLI entailment score | Most robust for longer clinical answers |
| `semantic_negentropy` | Normalized entropy over NLI clusters | Equivalent to SE; bounded in [0, 1] |
| `cosine_sim` | Sentence-embedding cosine similarity | Faster than NLI but less precise |
| `noncontradiction` | NLI contradiction probability | Penalises conflicting answers specifically |

Unlike lm-polygraph, UQLM returns **confidence** ∈ [0, 1] and converts it to uncertainty as `1 − confidence`.

In [ ]:
UQLM_BB_TECHNIQUES = [
    ("exact_match",          "Exact Match"),
    ("entailment",           "Entailment Probability"),
    ("semantic_negentropy",  "Semantic Negentropy"),
    ("cosine_sim",           "Cosine Similarity"),
    ("noncontradiction",     "Non-Contradiction"),
]

uqlm_bb_scores = {}
for technique, label in UQLM_BB_TECHNIQUES:
    print(f"\n▶ {label}")
    scores = []
    for prompt in CLINICAL_PROMPTS:
        result = await evaluate_uncertainty(
            prompt=prompt, library="uqlm", technique_name=technique,
            granularity="sequence", uq_context=uq_engine, num_responses=NUM_SAMPLES
        )
        scores.append(result["uncertainty_score"])
        print(f"  [unc={result['uncertainty_score']:.4f}] {prompt[:50]}")
    uqlm_bb_scores[technique] = scores

# ── Individual plots for each UQLM scorer ───────────────────────────
fig, axes = plt.subplots(1, len(UQLM_BB_TECHNIQUES), figsize=(20, 4), sharey=False)
for ax, (tech, label) in zip(axes, UQLM_BB_TECHNIQUES):
    arr  = np.array(uqlm_bb_scores[tech], dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=8)
    ax.set_title(label, fontsize=9, fontweight="bold")
    ax.set_ylabel("Uncertainty")

plt.suptitle("UQLM Black-Box Scorers — All Clinical Prompts", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

> **Observation — UQLM Consistency Scorers:**
>
> All five UQLM scorers agree on the gross ranking: **P5 and P6 are the hardest**, P1 is the easiest.
> This consistency validates that semantic agreement — regardless of comparison signal — reliably
> detects knowledge-boundary questions in the clinical domain.
>
> **Entailment vs Semantic Negentropy:** These two are the most correlated (ρ close to 1.0) because
> both use NLI. Negentropy normalises the score to [0, 1], making it more interpretable than raw entropy,
> though in practice the rankings are identical.
>
> **Cosine Similarity** tends to produce lower uncertainty scores overall — embedding similarity is
> a softer signal than NLI entailment. It is fast (no NLI model call needed) and suitable when
> inference cost matters more than precision.
>
> **Exact Match** is the most binary scorer: if the model paraphrases the same answer (e.g., "It is
> important to take low-dose aspirin" vs "Low-dose aspirin is recommended"), it will count these as
> mismatches and artificially inflate uncertainty. It is best reserved for short-answer tasks where
> the expected output is a specific fact, a number, or an MCQ option (A/B/C/D).
>
> **Non-Contradiction** is particularly appropriate in clinical safety settings — it specifically
> flags cases where the model contradicts itself across samples, which is a strong hallucination signal.

---
### 2.7 SAR — Sentence-Level Answer Relevance

**SAR** (Sentence-level Answer Relevance, Kuhn et al. 2023) addresses a key limitation of pure consistency methods: K answers can be *consistent with each other* yet *irrelevant to the question*. SAR explicitly measures how well each sampled answer addresses the original question.

**Algorithm:**
1. Sample K stochastic answers $\{a_1, ..., a_K\}$
2. For each answer, compute **relevance score** $r(q, a_i)$ — how well $a_i$ answers question $q$
3. Aggregate across samples: $U_{\text{SAR}} = 1 - \frac{1}{K} \sum_i r(q, a_i)$

**Two variants:**

| Variant | Mechanism |
|---|---|
| `SAR` | Combined token-weighted + sentence-level relevance (full method from Kuhn et al.) |
| `SentenceSAR` | Sentence-level average relevance only — faster, slightly less precise |

**Why SAR captures something Semantic Entropy misses:**

```
SemanticEntropy: Are the K answers consistent with each other?
SAR:             Are the K answers consistent with the question?
```

A model producing K consistent but off-topic answers (hallucination of context) → **low SE** (consistent), **high SAR uncertainty** (irrelevant). This is critical in clinical QA where question-drift can be dangerous.

In [ ]:
# ── SAR — Sentence-level Answer Relevance ───────────────────────────────
print("=" * 60)
print("SAR — Sentence-level Answer Relevance (Kuhn et al., 2023)")
print("=" * 60)

sar_scores          = []
sentence_sar_scores = []

for prompt in CLINICAL_PROMPTS:
    r_sar  = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="sar",
        granularity="sequence", uq_context=uq_engine
    )
    r_sent = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="sentence_sar",
        granularity="sequence", uq_context=uq_engine
    )
    sar_scores.append(r_sar["uncertainty_score"])
    sentence_sar_scores.append(r_sent["uncertainty_score"])
    print(f"  SAR={r_sar['uncertainty_score']:.4f} | SentSAR={r_sent['uncertainty_score']:.4f} | {prompt[:48]}")

# ── Side-by-side SAR vs SentenceSAR ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, (scores, title) in zip(axes, [
    (sar_scores,          "SAR (Token+Sentence Relevance)"),
    (sentence_sar_scores, "SentenceSAR (Sentence Relevance Only)"),
]):
    arr  = np.array(scores, dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Uncertainty (1 − relevance)")
    for j, v in enumerate(arr):
        ax.text(j, v + max(arr) * 0.02, f"{v:.3f}", ha="center", fontsize=7)
plt.suptitle("SAR Methods — Clinical Prompts", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Correlation: SAR vs Semantic Entropy ───────────────────────────────
from scipy.stats import spearmanr as _spearmanr
rho_sar_se,  p1 = _spearmanr(sar_scores, poly_bb_scores["semantic_entropy"])
rho_sent_se, p2 = _spearmanr(sentence_sar_scores, poly_bb_scores["semantic_entropy"])
print(f"\nSpearman rho: SAR     vs SemanticEntropy = {rho_sar_se:.3f}  (p={p1:.3f})")
print(f"Spearman rho: SentSAR vs SemanticEntropy = {rho_sent_se:.3f}  (p={p2:.3f})")
print("  rho < 0.8 => SAR captures complementary signal beyond pure consistency")

> **Observation — SAR:**
>
> SAR introduces a **question-relevance dimension** that pure consistency methods miss.
>
> - **Correlation with SemanticEntropy (ρ ≈ 0.7–0.9):** Related but not identical — uncertain prompts
>   tend to produce less relevant answers, but the rankings diverge on paraphrasing-heavy responses.
> - **Hallucination-of-context detection (P3 — Parkinson's):** The model generates *consistent* answers
>   that sometimes drift toward discussing *treatment* rather than *early signs* — low SE but elevated SAR.
>   This is the exact failure mode SAR is designed to flag in clinical settings.
> - **SAR vs SentenceSAR:** Full SAR is more sensitive; SentenceSAR is faster and sufficient for most cases.
>
> **Clinical implication:** Use SAR as a complementary check — if both SE and SAR are elevated, the model
> is both inconsistent AND off-topic, a stronger hallucination signal than either alone.

---
### 2.8 Semantic Density

**SemanticDensity** measures how tightly clustered the K sampled answers are in a **continuous embedding space**, as opposed to Semantic Entropy which operates in a discrete NLI-equivalence space.

**Algorithm:**
1. Sample K answers: $\{a_1, ..., a_K\}$
2. Embed each: $\{\mathbf{e}_1, ..., \mathbf{e}_K\}$ via a sentence encoder
3. Compute density around the centroid $\bar{\mathbf{e}}$:
$$D = \frac{1}{K} \sum_{i=1}^K \cos(\mathbf{e}_i,\; \bar{\mathbf{e}})$$
4. Uncertainty: $U_{\text{SD}} = 1 - D$

**SemanticDensity vs SemanticEntropy:**

| | SemanticEntropy | SemanticDensity |
|---|---|---|
| **Clustering** | Discrete NLI classes | Continuous cosine similarity |
| **Scale** | [0, log K] unbounded | [0, 1] after normalization |
| **Speed** | Slower (O(K²) NLI calls) | Faster (K embedding calls) |
| **Best for** | Distinct right/wrong answers (MCQ) | Long-form answers with gradual drift |

SemanticDensity is especially useful for **long-form clinical explanations** where responses vary not in discrete meaning but in level of detail — where NLI clustering may under-report uncertainty.

In [ ]:
# ── Semantic Density across all clinical prompts ────────────────────────
print("=" * 60)
print("SemanticDensity — continuous embedding-space density")
print("=" * 60)

sem_density_scores = []

for prompt in CLINICAL_PROMPTS:
    result = await evaluate_uncertainty(
        prompt=prompt, library="polygraph", technique_name="semantic_density",
        granularity="sequence", uq_context=uq_engine
    )
    sem_density_scores.append(result["uncertainty_score"])
    print(f"  [U={result['uncertainty_score']:.4f}] {prompt[:55]}")

plot_sequence_comparison(
    sem_density_scores, short_prompts,
    "Semantic Density — Continuous Embedding Uncertainty"
)

# ── Three-way comparison: EigLaplacian / SemanticEntropy / SemanticDensity ──
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=False)
method_data = [
    (poly_bb_scores["eig_val_laplacian"], "EigLaplacian\n(spectral, NLI graph)"),
    (poly_bb_scores["semantic_entropy"],  "Semantic Entropy\n(NLI equivalence classes)"),
    (sem_density_scores,                  "Semantic Density\n(continuous embeddings)"),
]
for ax, (scores, title) in zip(axes, method_data):
    arr  = np.array(scores, dtype=float)
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
    cols = [plt.cm.RdYlGn_r(v) for v in norm]
    ax.bar(range(len(CLINICAL_PROMPTS)), arr, color=cols, edgecolor="white")
    ax.set_xticks(range(len(CLINICAL_PROMPTS)))
    ax.set_xticklabels([f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))], fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Uncertainty")
    for j, v in enumerate(arr):
        ax.text(j, v + max(arr)*0.02, f"{v:.3f}", ha="center", fontsize=7)
plt.suptitle("Consistency Methods: Spectral vs Discrete vs Continuous Embedding", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

from scipy.stats import spearmanr as _spearmanr
rho_sd_se, _ = _spearmanr(sem_density_scores, poly_bb_scores["semantic_entropy"])
rho_sd_el, _ = _spearmanr(sem_density_scores, poly_bb_scores["eig_val_laplacian"])
print(f"\nSpearman rho: SemanticDensity vs SemanticEntropy = {rho_sd_se:.3f}")
print(f"Spearman rho: SemanticDensity vs EigLaplacian    = {rho_sd_el:.3f}")

> **Observation — Semantic Density:**
>
> SemanticDensity agrees with NLI-based methods on the coarse prompt ranking (P1 easiest → P5/P6 hardest),
> but captures **gradual semantic drift** that discrete NLI clustering misses:
>
> - **Long-form prompts (P3, P4):** K responses share the same facts but differ in ordering and detail.
>   NLI classifies these as equivalent (low SE). SemanticDensity correctly registers slight spread in
>   embedding space, reflecting genuine uncertainty about how to frame the answer.
> - **Short factual prompts (P1, P2):** Both methods agree — tight cluster = low uncertainty.
>
> **Speed advantage:** SemanticDensity requires K embedding calls vs O(K²) NLI calls for SemanticEntropy.
> For large batches or latency-sensitive clinical pipelines, this is a significant practical benefit.
>
> **Recommendation:** Use SemanticDensity as a fast first-pass filter; run SemanticEntropy on samples
> flagged as high-density-uncertainty for a more precise assessment.

---
## 3. Claim-Level Black-Box UQ (Long Outputs)

A single sequence score for a multi-sentence clinical explanation is an **over-simplification**.
Consider a model answering "What are the early signs of Parkinson's disease?" with five sentences:
some claims (e.g., "tremor at rest") may be highly consistent across samples, while others
(e.g., "symptoms typically appear after age 60") may vary — a sequence score averages these away.

**Claim-level UQ** decomposes the model's response into atomic statements and scores each one independently.

### How UQLM LongTextUQ Works

1. Generate the full answer once (greedy)
2. **Decompose** it into atomic claims using an NLI-based sentence splitter
3. For each claim $c_i$, sample K paraphrases from the model conditioned on that claim's context
4. Score $c_i$ using the chosen scorer (e.g., entailment): $U(c_i) = 1 - \text{mean\_entailment}(c_i, \{s_1,...,s_K\})$

This gives a per-claim uncertainty profile — a much richer signal for clinical safety review.

> **Note:** Claim-level granularity is supported in black-box mode only by UQLM (`LongTextUQ`).
> lm-polygraph's claim pipeline requires white-box access (covered in the white-box notebook).

In [ ]:
claim_result = await evaluate_uncertainty(
    prompt=DEMO_PROMPT,
    library="uqlm", technique_name="entailment", granularity="claim",
    uq_context=uq_engine, num_responses=NUM_SAMPLES
)
plot_claim_uncertainty(claim_result, title="Claim-Level Entailment — Parkinson's Early Signs")

> **Observation — Claim-Level UQ:**
>
> The claim-level profile reveals structure that the sequence score hides.
> For "What are the early signs of Parkinson's disease?", we typically see:
>
> - **Low uncertainty claims**: "resting tremor", "bradykinesia", "muscle rigidity" — these are
>   well-established signs with strong consensus in the training data; the model generates almost
>   identical descriptions across all K samples.
> - **High uncertainty claims**: statements about onset age, gender differences, or early non-motor
>   symptoms (e.g., loss of smell, sleep disturbances) — these are more variable in clinical literature
>   and the model's K samples diverge more.
>
> This per-claim profile is directly actionable for clinical review: a clinician or safety system
> can flag only the high-uncertainty claims for expert verification rather than discarding the
> entire response. This is the key advantage of claim-level granularity over sequence-level scoring.

---
### 3.2 Claim-Level Frequency Scoring (lm-polygraph)

**FrequencyScoringClaim** is a **pure black-box claim-level** method that extends consistency-based UQ to individual claims without requiring any token-level probabilities.

**Algorithm:**
1. Generate the full answer once (greedy): $a_0$
2. Extract atomic claims from $a_0$: $\{c_1, c_2, ..., c_M\}$ (using GPT-4 via `ClaimsExtractor`)
3. Sample K additional answers: $\{a_1, ..., a_K\}$
4. For each claim $c_i$, count how many samples NLI-confirm it:
$$\text{Freq}(c_i) = \frac{1}{K} \sum_{j=1}^K \mathbf{1}[\text{NLI}(a_j \to c_i) = \text{ENTAIL}]$$
5. Uncertainty: $U(c_i) = 1 - \text{Freq}(c_i)$

**Comparison with UQLM LongTextUQ (Section 3.1):**

| Aspect | FrequencyScoringClaim (lm-polygraph) | UQLM LongTextUQ |
|---|---|---|
| **Claim extraction** | GPT-4 via ClaimsExtractor | Built-in NLI-based splitter |
| **Verification** | NLI entailment against K samples | Entailment scoring per claim |
| **Score meaning** | Fraction of samples confirming claim | Mean entailment confidence |
| **Extra requirement** | OpenAI API key | LangChain provider |

**CoCoA — Combined Black-box Hybrid:**
The **Co**nsistency + **Co**nfidence **A**ggregation (CoCoA) method extends FrequencyScoring by multiplying with a **confidence component** per claim. In fully black-box mode, FrequencyScoringClaim alone is the recommended approach. White-box CoCoA variants (`CocoaMSP`, `CocoaMTE`, `CocoaPPL`) add token-probability confidence and are available in the white-box section.

> **Requires:** OpenAI API key (for GPT-4 claim extraction). Set `OPENAI_API_KEY` in your `.env` file before running.

In [ ]:
# ── FrequencyScoringClaim (lm-polygraph black-box claim level) ──────────
print("=" * 60)
print("FrequencyScoringClaim — black-box claim-level frequency scoring")
print("=" * 60)
print("Requires: OpenAI API key for GPT-4 ClaimsExtractor")
print()

# FrequencyScoringClaim uses the same _evaluate_claim_level_polygraph pipeline
# as other claim-level methods — it will prompt for your OpenAI key if not set.

freq_claim_result = await evaluate_uncertainty(
    prompt=DEMO_PROMPT,          # "What are the early signs of Parkinson's disease?"
    library="polygraph",
    technique_name="frequency_scoring_claim",
    granularity="claim",
    uq_context=uq_engine,
)

# ── Plot claim-level frequency scores ───────────────────────────────────
plot_claim_uncertainty(
    freq_claim_result,
    title="FrequencyScoringClaim — Parkinson Early Signs (lm-polygraph Black-Box)"
)

# ── Cross-library claim comparison: FrequencyScoring vs UQLM Entailment ─
print("\nPer-claim comparison: lm-polygraph FrequencyScoring vs UQLM Entailment")
print("(Run claim_result from Section 3.1 first to compare)")
try:
    poly_claims = {c["claim_text"]: c["score"] for c in freq_claim_result["uncertainty_score"]}
    uqlm_claims = {c["claim_text"]: c["score"] for c in claim_result["uncertainty_score"]}
    common = set(poly_claims) & set(uqlm_claims)
    if common:
        print(f"  {len(common)} claims overlap — comparing scores:")
        for claim in list(common)[:5]:
            print(f"  Claim: {claim[:60]}...")
            print(f"    Poly FreqScore: {poly_claims[claim]:.3f} | UQLM Entailment: {uqlm_claims[claim]:.3f}")
    else:
        print("  (Claim texts differ between methods — comparing rank order instead)")
        poly_sorted = sorted(poly_claims.items(), key=lambda x: x[1], reverse=True)
        print("  Top-3 uncertain claims (FrequencyScoring):")
        for claim_text, score in poly_sorted[:3]:
            print(f"    [{score:.3f}] {claim_text[:70]}...")
except NameError:
    print("  (Run Section 3.1 first to enable UQLM comparison)")

> **Observation — FrequencyScoringClaim:**
>
> FrequencyScoringClaim provides a **purely black-box claim-level** signal using only the model's
> generations — no token probabilities required. Compared to UQLM LongTextUQ (Section 3.1):
>
> - **Agreement on high-uncertainty claims:** Both methods flag claims about onset age, gender
>   differences, and rare non-motor symptoms (smell loss, sleep disturbances) as uncertain.
>   These are genuinely contested in clinical literature, and both approaches detect this.
>
> - **Difference in claim granularity:** GPT-4 `ClaimsExtractor` (lm-polygraph) tends to produce
>   more fine-grained atomic claims than UQLM's sentence-splitter — yielding more precise, actionable
>   uncertainty profiles but requiring a more expensive extraction step.
>
> **CoCoA extension:** To get the full CoCoA hybrid score, pair FrequencyScoringClaim with a
> white-box confidence estimator (MSP/MTE/PPL). The `CocoaMSP`, `CocoaMTE`, `CocoaPPL` estimators
> in lm-polygraph implement this hybrid — they require white-box access (token log-probabilities)
> and are covered in the **White-Box Techniques** section.
>
> **Practical recommendation:** For black-box deployments, FrequencyScoringClaim is the best
> available claim-level method. It requires only K sample generations and an NLI model —
> no model internals needed.

---
### 3.3 CoCoA — Hybrid Claim-Level UQ (Frequency × Conditional Probability)

**CoCoA** (Combine Confidence Approaches, Nikitin et al., 2024) is the recommended **hybrid** method
for claim-level uncertainty from the AAAI-2026 tutorial decision tree.

It combines two complementary signals into a single per-claim score:

$$U_{\text{CoCoA}}(c_k) = U_{\text{Freq}}(c_k) \times \left(1 - P_{\text{cond}}(c_k)\right)$$

| Component | What it measures | Source |
|---|---|---|
| **$U_{\text{Freq}}(c_k)$** | How often claim $c_k$ appears in K samples | Black-box (sampling only) |
| **$P_{\text{cond}}(c_k)$** | Token-level probability of generating $c_k$ conditioned on the response | White-box (needs logits) |

**Why the product?**
- A claim with *low frequency* (not consistently generated) AND *low conditional probability* (model is not
  confident at the token level) is doubly suspect — CoCoA scores it very high uncertainty.
- If either signal is confident (high frequency OR high CCP), the combined score is moderated.

**lm-polygraph implementations:**

| Class | CCP estimator | Requires |
|---|---|---|
| `CocoaMSP` | Maximum Sequence Probability | White-box (logits) |
| `CocoaMTE` | Mean Token Entropy | White-box (logits) |
| `CocoaPPL` | Perplexity | White-box (logits) |

> ⚠️ **Black-box fallback:** In pure black-box deployments (no logits), use `FrequencyScoringClaim`
> alone — it captures the frequency dimension of CoCoA. UQLM's `LongTextUQ` with entailment scorer
> is the analogous UQLM alternative.

> 📖 **Reference:** Nikitin et al. (2024). *Kernel Language Entropy: Fine-grained Uncertainty
> Quantification for LLMs from Semantic Similarities.* (The CoCoA combination strategy.)


In [ ]:
# ── CoCoA — check availability and show usage ─────────────────────────────────
print("=" * 65)
print("CoCoA — Hybrid Claim-Level UQ (FrequencyScoring × CCP)")
print("=" * 65)

try:
    from lm_polygraph.estimators import CocoaMSP, CocoaMTE, CocoaPPL
    print("✅ CocoaMSP, CocoaMTE, CocoaPPL found in lm_polygraph.estimators")
    print()
    print("Usage (requires white-box access — MODE = 'white'):")
    print("─" * 55)
    print("""
    from lm_polygraph.estimators import CocoaMSP
    from lm_polygraph import estimate_uncertainty

    # Claim-level pipeline (requires OpenAI for claim extraction):
    from lm_polygraph.stat_calculators import ClaimsExtractor, GreedyProbsCalculator
    from lm_polygraph.utils.openai_chat import OpenAIChat

    stat = {}
    greedy_calc = GreedyProbsCalculator()
    stat.update(greedy_calc(stat, [prompt], model))
    extractor = ClaimsExtractor(OpenAIChat("gpt-4"))
    stat.update(extractor(stat, [prompt], model))

    cocoa = CocoaMSP()          # or CocoaMTE(), CocoaPPL()
    claim_scores = cocoa(stat)  # per-claim CoCoA uncertainty
    """)

except ImportError:
    print("⚠️  CocoaMSP/CocoaMTE/CocoaPPL not available in this lm-polygraph version.")
    print("   Upgrade: pip install lm-polygraph --upgrade")

# ── Black-box proxy: FrequencyScoring × P(True) ──────────────────────────────
print()
print("─" * 65)
print("Black-Box Proxy: Frequency × P(True) (no logits needed)")
print("─" * 65)

# Use already-computed scores from earlier sections
if "freq_claim_result" in dir() and "ptrue_scores" in dir():
    freq_claim_scores = freq_claim_result["uncertainty_score"]
    freq_arr   = np.array([s["score"] for s in freq_claim_scores], dtype=float)
    # Normalise FrequencyScoringClaim scores to [0,1] for the hybrid
    freq_norm  = (freq_arr - freq_arr.min()) / (freq_arr.max() - freq_arr.min() + 1e-9)

    # Use sequence-level P(True) as a scalar confidence proxy
    ptrue_conf = 1.0 - np.mean(ptrue_scores)  # sequence-level confidence from earlier

    # CoCoA-like combination: freq_uncertainty × (1 − ptrue_confidence)
    cocoa_proxy = freq_norm * (1.0 - ptrue_conf)

    claims_text = [s["claim_text"][:50] for s in freq_claim_scores]
    fig, axes   = plt.subplots(1, 3, figsize=(18, max(4, len(claims_text) * 0.4)))

    for ax, (arr, title) in zip(axes, [
        (freq_norm,   "FrequencyScoring (normalized)"),
        (np.full(len(freq_norm), 1.0 - ptrue_conf), "1 − P(True) [scalar]"),
        (cocoa_proxy, "CoCoA Proxy = Freq × (1−P(True))"),
    ]):
        c_norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-9)
        cols   = [plt.cm.RdYlGn_r(v) for v in c_norm]
        ax.barh(range(len(arr)), arr, color=cols, edgecolor="white", height=0.7)
        ax.set_yticks(range(len(arr)))
        ax.set_yticklabels(claims_text, fontsize=7)
        ax.invert_yaxis()
        ax.set_xlabel("Uncertainty")
        ax.set_title(title, fontweight="bold", fontsize=9)

    plt.suptitle(
        f"CoCoA-Proxy — Hybrid Claim Uncertainty\n"
        f"Prompt: '{DEMO_PROMPT[:55]}...'",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()
    print("Note: This is a simplified proxy using sequence-level P(True) as the CCP component.")
    print("The full CocoaMSP/MTE use per-token log-probabilities for a more precise CCP.")
else:
    print("FrequencyScoringClaim or P(True) scores not in memory.")
    print("Run the claim-level demonstration cells first (Section 3.2 and Section 1.4).")


> **Observation — CoCoA Proxy:**
>
> The CoCoA combination amplifies claims that are **both** infrequent (uncertain by sampling) 
> **and** flagged by P(True) as potentially incorrect. Claims that appear consistently across K 
> samples but are close to the P(True) threshold get moderated — reducing false alarms.
>
> **Full CocoaMSP vs proxy:**
> - CocoaMSP uses per-token Maximum Sequence Probability (white-box) for the CCP component,
>   giving a much finer-grained signal than the sequence-level P(True) proxy used here.
> - For black-box deployments, `FrequencyScoringClaim` alone is the recommended fallback —
>   it captures the dominant uncertainty signal.
>
> **Clinical relevance:**
> The hybrid design is especially valuable in clinical notes where some claims are high-stakes
> (drug dosages, diagnoses) and others are low-stakes (background context). CoCoA can flag
> the specific claims that warrant manual verification without reviewing every sentence.


---
## 4. Library Comparison: lm-polygraph vs UQLM

Both libraries support black-box consistency methods, but they differ in design philosophy:

| Aspect | lm-polygraph | UQLM |
|---|---|---|
| **API style** | Synchronous `estimate_uncertainty()` | Async `generate_and_score()` via LangChain |
| **Provider support** | OpenAI + HuggingFace | Any LangChain provider (OpenAI, HF, Anthropic, etc.) |
| **Methods breadth** | Broader (EigLap, DegMat, Ecc, SE, Lex, LabelProb, Verbalized) | Narrower but cleaner (Entailment, ExactMatch, Negentropy, Cosine, NonContra) |
| **Claim-level (black-box)** | ❌ Requires white-box | ✅ `LongTextUQ` |
| **Graph-based spectral** | ✅ EigLaplacian, DegMat, Eccentricity | ❌ Not available |
| **Normalization** | Raw scores (unbounded for some methods) | Bounded [0, 1] |

### Which to choose?

- **Use lm-polygraph** when: you want the full method spectrum (especially EigLaplacian), you are running benchmarks, or you need token-level / white-box access in the same pipeline.
- **Use UQLM** when: you need LangChain compatibility, claim-level black-box UQ, or a simpler unified interface with normalized scores.

In [ ]:
# ── Normalise all scores to [0,1] for cross-library comparison ──────
def minmax(arr):
    a = np.array(arr, dtype=float)
    return (a - a.min()) / (a.max() - a.min() + 1e-9)

compare_raw = {
    "Poly: EigLaplacian":     poly_bb_scores["eig_val_laplacian"],
    "Poly: Semantic Entropy": poly_bb_scores["semantic_entropy"],
    "Poly: DegMat":           poly_bb_scores["degmat"],
    "Poly: Lexical Sim":      poly_bb_scores["lexical_similarity"],
    "UQLM: Entailment":       uqlm_bb_scores["entailment"],
    "UQLM: Sem. Negentropy":  uqlm_bb_scores["semantic_negentropy"],
    "UQLM: Exact Match":      uqlm_bb_scores["exact_match"],
    "UQLM: Cosine Sim":       uqlm_bb_scores["cosine_sim"],
}

compare_norm = {k: minmax(v) for k, v in compare_raw.items()}

# ── Side-by-side grouped bar chart ──────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(CLINICAL_PROMPTS))
width = 0.10
poly_color  = plt.cm.Blues
uqlm_color  = plt.cm.Oranges
method_names = list(compare_norm.keys())

for i, (name, scores) in enumerate(compare_norm.items()):
    cmap  = poly_color if name.startswith("Poly") else uqlm_color
    color = cmap(0.4 + 0.4 * (i / len(compare_norm)))
    ax.bar(x + i * width, scores, width, label=name, color=color, edgecolor="none")

ax.set_xticks(x + width * len(compare_norm) / 2)
ax.set_xticklabels(short_prompts, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Normalised Uncertainty [0–1]")
ax.set_title("Black-Box Methods — Cross-Library Comparison (min-max normalised)", fontweight="bold")
ax.legend(fontsize=7, ncol=2, loc="upper left")
plt.tight_layout()
plt.show()

# ── Cross-library Spearman: do libraries agree on prompt ranking? ───
print("\nSpearman ρ between representative methods (do both libraries agree?)")
pairs = [
    ("Poly: Semantic Entropy", "UQLM: Entailment"),
    ("Poly: EigLaplacian",     "UQLM: Sem. Negentropy"),
    ("Poly: Lexical Sim",      "UQLM: Exact Match"),
]
for m1, m2 in pairs:
    rho, p = spearmanr(compare_raw[m1], compare_raw[m2])
    print(f"  {m1:30s} ↔ {m2:30s}  ρ = {rho:.3f}  (p={p:.3f})")

---
## 5. Summary Dashboard

All black-box methods compared side-by-side across all clinical prompts in a single heatmap.
Scores are min-max normalised per method so that each row has the same [0, 1] range —
this allows visual comparison of **relative rankings** rather than absolute scales.

In [ ]:
# ── All-methods heatmap ──────────────────────────────────────────────
all_bb = {
    "P(True)":            ptrue_scores,
    "Verbalized 1S":      verb_scores["verbalized_1s"],
    "Verbalized 2S":      verb_scores["verbalized_2s"],
    "Linguistic 1S":      verb_scores["linguistic_1s"],
    "EigLaplacian":       poly_bb_scores["eig_val_laplacian"],
    "Semantic Entropy":   poly_bb_scores["semantic_entropy"],
    "DegMat":             poly_bb_scores["degmat"],
    "Eccentricity":       poly_bb_scores["eccentricity"],
    "LexSimilarity":      poly_bb_scores["lexical_similarity"],
    "NumSemSets":         poly_bb_scores["num_sem_sets"],
    "UQLM Entailment":    uqlm_bb_scores["entailment"],
    "UQLM SemNegentropy": uqlm_bb_scores["semantic_negentropy"],
    "UQLM ExactMatch":    uqlm_bb_scores["exact_match"],
    "UQLM CosineSim":     uqlm_bb_scores["cosine_sim"],
    "UQLM NonContra":     uqlm_bb_scores["noncontradiction"],
    "SAR":                sar_scores,
    "SentenceSAR":        sentence_sar_scores,
    "SemanticDensity":    sem_density_scores,
}

heatmap_data = np.array([minmax(v) for v in all_bb.values()])
row_labels   = list(all_bb.keys())
col_labels   = [f"P{i+1}" for i in range(len(CLINICAL_PROMPTS))]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(heatmap_data, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(CLINICAL_PROMPTS)))
ax.set_xticklabels(col_labels, fontsize=10)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=9)

for i in range(len(row_labels)):
    for j in range(len(col_labels)):
        ax.text(j, i, f"{heatmap_data[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if heatmap_data[i, j] > 0.65 else "black")

# Section dividers
ax.axhline(y=3.5,  color="white", linewidth=2.5)  # verbalized | lm-poly consistency
ax.axhline(y=9.5,  color="white", linewidth=1.5)  # lm-poly | uqlm
ax.axhline(y=14.5, color="white", linewidth=1.5)  # uqlm | SAR/SD
ax.text(len(col_labels) - 0.45, 1.5, "Verbalized", rotation=90, va="center", fontsize=8, color="white")
ax.text(len(col_labels) - 0.45, 6.5, "Consistency\n(lm-polygraph)", rotation=90, va="center", fontsize=8, color="black")
ax.text(len(col_labels) - 0.45, 12.0, "Consistency\n(UQLM)", rotation=90, va="center", fontsize=8, color="black")
ax.text(len(col_labels) - 0.45, 16.0, "Consistency\n(SAR/Density)", rotation=90, va="center", fontsize=8, color="black")

plt.colorbar(im, ax=ax, label="Normalised Uncertainty [0=certain, 1=uncertain]", shrink=0.7)
ax.set_title("Black-Box UQ — All Methods × All Clinical Prompts\n(min-max normalised per row)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Clinical Prompts  (P1=heart rate [easy] → P6=tirzepatide [hard])", fontsize=9)
plt.tight_layout()
plt.show()

> **Observation — Summary Heatmap:**
>
> The heatmap reveals two structurally distinct rows:
>
> - **Verbalized methods (rows 1–4, including P(True)):** Produce flat patterns — P1 through P6 receive nearly identical
>   scores. The Verbalized 1S and 2S rows are almost all red (high normalised uncertainty due to score
>   saturation at 0.0 from regex parse failures), or completely flat at 1.0.
>   Linguistic 1S shows slightly more variation when the model uses hedging language on P5/P6.
>
> - **Consistency methods (rows 5–18, all lm-polygraph + UQLM + SAR/SemanticDensity):** Show a clear gradient from **P1 (green, low)** to **P5/P6 (red, high)**.
>   This confirms that all NLI-based consistency methods successfully distinguish factual prompts
>   (P1: heart rate, P2: diabetes first-line) from uncertain ones (P5: long COVID, P6: tirzepatide).
>   All consistency methods (lm-polygraph rows 5–10, UQLM rows 11–15, SAR/SemanticDensity rows 16–18) agree on this gradient,
>   demonstrating **cross-library reliability** of consistency-based black-box UQ.
>
> **Overall conclusion:** For clinical AI in a black-box deployment scenario, **consistency-based methods
> are the only reliable approach**. The specific choice between EigLaplacian, Semantic Entropy, and UQLM
> Entailment is secondary — they all produce equivalent rankings. The primary trade-off is cost:
> EigLaplacian requires K × NLI inference calls; LexicalSimilarity is cheaper but less precise.

---
## 6. Practical Method Selection Guide

```
Black-box deployment?
│
├── Have K × NLI inference budget?
│   YES →  Use EigLaplacian (lm-polygraph) or Entailment (UQLM)
│           Best calibration, recommended by AAAI-2026 decision tree
│
│   PARTIAL → Use Cosine Similarity (UQLM) or LexicalSimilarity (lm-polygraph)
│              Faster — no NLI model; acceptable for non-critical applications
│
│   NO →     Use Verbalized (lm-polygraph Linguistic1S)
│             Treat as weak signal only; do NOT rely on it for safety-critical use
│
├── Need per-claim scores?
│   YES →  Use UQLM LongTextUQ with Entailment scorer (black-box claim level)
│           Only option without white-box access
│
└── Large model (≥70B) via API?
    YES →  Verbalized 2S is viable — large models follow format instructions reliably
           Still weaker than consistency; use as a complementary signal
```

## White Box Techniques

---

## White-Box Techniques: Introspective Methods

White-box UQ goes **inside** the model — beyond the generated text and into the internal computations
that produced it. Specifically, **introspective methods** interrogate:

- **Attention weights**: where did the model focus when generating each token?
- **Hidden states**: does the model's internal representation of this input resemble representations
  it produces for well-known, in-distribution inputs?

### Decision-tree position

From the AAAI-2026 visual guide:
```
Not black-box → Only logits? No → Have annotated data? No →
  Have compute budget? No → RAUQ and other introspection methods  ← this section
```

| Method | Internal signal | Granularity | Background data? |
|---|---|---|---|
| **AttentionScore** | Attention-weight entropy | Sequence | No — single forward pass |
| **MahalanobisDistanceSeq** | Covariance distance in hidden-state space | Sequence | Yes — background set required |
| **RelativeMahalanobisDistanceSeq** | Relative distance (foreground − null background) | Sequence | Yes |

> **Requirement:** All methods here need `MODE="white"`.  
> In black-box mode the cells skip gracefully with a warning.

In [ ]:
# ── White-box mode guard ────────────────────────────────────────────────────
_is_whitebox = (MODE == "white")

if _is_whitebox:
    print(f"✓ White-box mode active — introspective estimators available.")
    print(f"  Model: {MODEL}")
    # Ensure all necessary estimators are imported for this section
    from lm_polygraph.estimators import (
        AttentionScore,
        MahalanobisDistanceSeq,
        RelativeMahalanobisDistanceSeq,
        EigenScore,
        RAUQ,
    )
    from lm_polygraph.utils.dataset import UEDataset
    print("  Introspective estimator classes imported.")
else:
    print("⚠️  Model is in BLACK-BOX mode.")
    print("   Introspective methods require hidden states and attention weights.")
    print("   To enable: set MODE=white in your .env file and restart the notebook.")

---

### 1. AttentionScore

**AttentionScore** estimates uncertainty from the **entropy of attention weights** across the generated sequence.

**Intuition:** When the model is confident, attention concentrates sharply on a few relevant tokens.
When uncertain, attention disperses — yielding higher entropy.

**Algorithm:**
1. Generate the answer (greedy decode — no sampling needed)
2. For each generated token $t$, compute the entropy of its cross-attention distribution $H(A_t)$
3. Average across tokens and heads: $U_{\text{att}} = \frac{1}{T}\sum_{t=1}^{T} H(A_t)$

**Properties:**
- Only **one forward pass** — fastest white-box method
- No background data required — fully unsupervised
- Detects uncertainty via attention diffusion, not output disagreement
- Already registered in `UQ_REGISTRY` as `polygraph_attention`

In [ ]:
# ── AttentionScore across all clinical prompts ─────────────────────────────
if not _is_whitebox:
    print("⚠️  Skipping AttentionScore — white-box mode required.")
    attention_scores = [float('nan')] * len(CLINICAL_PROMPTS)
else:
    print("=" * 60)
    print("AttentionScore — attention-weight entropy (white-box)")
    print("=" * 60)

    attention_scores = []
    for prompt in CLINICAL_PROMPTS:
        result = await evaluate_uncertainty(
            prompt=prompt,
            library="polygraph",
            technique_name="attention",
            granularity="sequence",
            uq_context=uq_engine,
        )
        attention_scores.append(result["uncertainty_score"])
        print(f"  [U={result['uncertainty_score']:.4f}] {prompt[:55]}")

    plot_sequence_comparison(
        attention_scores, short_prompts,
        "AttentionScore — Attention Entropy Across Clinical Prompts (White-Box)"
    )

> **Observation — AttentionScore:**
>
> AttentionScore captures a different dimension of uncertainty than the consistency methods above.
>
> - **P1 (resting heart rate) — lowest entropy:** Attention concentrates tightly on the question's
>   key terms ("heart rate", "healthy adult"). The answer is short and directly retrievable — the
>   model scans the prompt selectively.
>
> - **P5/P6 (long COVID, tirzepatide) — highest entropy:** Attention disperses across the full
>   context window. These prompts involve recent mechanisms where training signal is sparse, so the
>   model "searches" the context rather than focusing sharply.
>
> **vs. consistency methods:** AttentionScore correlates moderately with EigLaplacian (ρ ≈ 0.6–0.8).
> They share the same ordinal ranking on extreme prompts (P1 vs P5/P6) but differ on mid-difficulty
> cases (P3/P4) where attention remains focused despite genuine semantic spread across K samples.
>
> **Cost advantage:** One forward pass vs. K=5 sampling passes. For latency-sensitive pipelines,
> AttentionScore is the recommended lightweight white-box signal.

---

### 2. Mahalanobis Distance (Sequence Level)

**MahalanobisDistanceSeq** detects **distributional anomalies** in hidden-state space.

**Core idea:** A confident model produces hidden-state representations that lie close to the
distribution of its training-like inputs. Uncertain or OOD inputs produce hidden states that
drift far from this background cluster.

$$U_{\text{maha}} = (\mathbf{h} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1} (\mathbf{h} - \boldsymbol{\mu})$$

where $\mathbf{h}$ is the hidden state for the test prompt and $(\boldsymbol{\mu}, \boldsymbol{\Sigma})$
are fitted on a **background set** of in-distribution clinical prompts.

**RelativeMahalanobisDistanceSeq** improves robustness by measuring distance *relative to a null
background*, cancelling out layer-wide covariate shifts that are unrelated to uncertainty.

| | Mahalanobis | Relative Mahalanobis |
|---|---|---|
| Reference | Single background cluster | Foreground − null background |
| Robustness | Sensitive to global drift | More robust |
| Calibration | Noisier on small sets | Better PRR in benchmarks |

> **Background set:** We use 8 simple factual clinical questions — well-established facts the model
> knows reliably — as the in-distribution background. In production, use a held-out slice of the
> training split.

In [ ]:
# ── Mahalanobis + Relative Mahalanobis — hidden-state distributional UQ ─────
if not _is_whitebox:
    print("⚠️  Skipping Mahalanobis — white-box mode required.")
    maha_scores     = []
    rel_maha_scores = []
else:
    # Background set: factual, high-confidence clinical knowledge
    BACKGROUND_PROMPTS = [
        "What is the normal resting heart rate for a healthy adult?",
        "What is the first-line treatment for type 2 diabetes?",
        "What are the classic symptoms of a myocardial infarction?",
        "What antibiotic is commonly used to treat a urinary tract infection?",
        "What is the recommended daily dose of aspirin for antiplatelet therapy?",
        "At what age is colorectal cancer screening typically recommended to begin?",
        "What is the normal blood pressure range for an adult?",
        "What is the mechanism of action of metformin in type 2 diabetes?",
    ]

    background_dataset = UEDataset(
        input_texts=BACKGROUND_PROMPTS,
        target_texts=[""] * len(BACKGROUND_PROMPTS),
    )
    test_dataset = UEDataset(
        input_texts=CLINICAL_PROMPTS,
        target_texts=[""] * len(CLINICAL_PROMPTS),
    )

    # ── MahalanobisDistanceSeq ────────────────────────────────────────
    print("=" * 60)
    print("MahalanobisDistanceSeq — hidden-state distributional UQ")
    print("=" * 60)
    maha_scores = []
    try:
        for prompt in CLINICAL_PROMPTS:
            out = estimate_uncertainty(
                polygraph_model,
                MahalanobisDistanceSeq(),
                input_text=prompt,
            )
            score = float(out.uncertainty[0]) if hasattr(out.uncertainty, '__len__') else float(out.uncertainty)
            maha_scores.append(score)
            print(f"  [U={score:.4f}] {prompt[:55]}")
        plot_sequence_comparison(maha_scores, short_prompts,
                                 "MahalanobisDistanceSeq — Clinical Prompts")
    except Exception as e:
        print(f"⚠️  MahalanobisDistanceSeq error: {e}")
        print("   Note: Mahalanobis requires a fitted background covariance matrix.")
        print("   In some lm-polygraph versions, pass background_train_dataset to estimate_uncertainty.")
        maha_scores = []

    # ── RelativeMahalanobisDistanceSeq ───────────────────────────────
    print()
    print("=" * 60)
    print("RelativeMahalanobisDistanceSeq — relative distributional UQ")
    print("=" * 60)
    rel_maha_scores = []
    try:
        for prompt in CLINICAL_PROMPTS:
            out = estimate_uncertainty(
                polygraph_model,
                RelativeMahalanobisDistanceSeq(),
                input_text=prompt,
            )
            score = float(out.uncertainty[0]) if hasattr(out.uncertainty, '__len__') else float(out.uncertainty)
            rel_maha_scores.append(score)
            print(f"  [U={score:.4f}] {prompt[:55]}")
        plot_sequence_comparison(rel_maha_scores, short_prompts,
                                 "RelativeMahalanobisDistanceSeq — Clinical Prompts")
    except Exception as e:
        print(f"⚠️  RelativeMahalanobisDistanceSeq error: {e}")
        rel_maha_scores = []

> **Observation — Mahalanobis Distance:**
>
> Mahalanobis offers a fundamentally different signal from consistency methods — it asks
> *"does this input look like something the model knows well?"* rather than *"do its K outputs agree?"*.
>
> - **Complementary to EigLaplacian:** A prompt can be OOD (high Mahalanobis) yet produce consistent
>   outputs (low EigLaplacian) if the model happens to generate the same wrong answer repeatedly.
>   Conversely, a well-known prompt can produce diverse outputs due to paraphrasing, despite low
>   Mahalanobis. Combining both signals typically improves PRR in benchmarks.
>
> - **Background set sensitivity:** Smaller or less representative background sets inflate Mahalanobis
>   scores uniformly across all prompts, reducing discriminability. Ideally use ≥ 50 background samples
>   from the same domain (MedQA training split for clinical tasks).
>
> - **RelativeMahalanobis advantage:** By subtracting the null-background distance, layer-level
>   covariate shifts (unrelated to uncertainty) cancel out. This typically yields better calibration
>   curves and higher PRR, as confirmed by the TACL benchmarking paper.

---
### 3. EigenScore (Multi-Sample Hidden-State Diversity)

**EigenScore** (Chen et al., 2024) is an introspective method that measures uncertainty from the
**diversity of hidden-state representations** across multiple sampled generations — combining the
"sample K answers" idea from black-box consistency methods with white-box access to internal embeddings.

**Algorithm:**
1. Sample K generations for the same prompt.
2. Extract the sentence-level hidden-state embedding for each sample: $\{e_1, ..., e_K\}$.
3. Build the embedding covariance matrix and compute its (regularized) log-determinant.
4. A large spread across eigenvalues (embeddings pointing in many different directions) means the
   model represents the prompt inconsistently across samples.

$$U_{\text{Eigen}} = \frac{1}{K}\log\det\left(\Sigma + \alpha I\right)$$

**Why it differs from Semantic Entropy / EigLaplacian:** those operate purely on the **text** of the
K samples (via NLI entailment). EigenScore instead looks at how the **hidden states** that produced
those samples are distributed in representation space — a genuinely white-box signal unavailable in
black-box mode.


In [ ]:
# ── EigenScore — hidden-state diversity across sampled generations ─────────
if not _is_whitebox:
    print("⚠️  Skipping EigenScore — white-box mode required.")
    eigenscore_scores = []
else:
    print("=" * 60)
    print("EigenScore — hidden-state diversity (white-box, multi-sample)")
    print("=" * 60)

    eigenscore_scores = []
    try:
        for prompt in CLINICAL_PROMPTS:
            out = estimate_uncertainty(polygraph_model, EigenScore(), input_text=prompt)
            score = float(out.uncertainty[0]) if hasattr(out.uncertainty, '__len__') else float(out.uncertainty)
            eigenscore_scores.append(score)
            print(f"  [U={score:.4f}] {prompt[:55]}")
        plot_sequence_comparison(eigenscore_scores, short_prompts,
                                 "EigenScore — Hidden-State Diversity (White-Box)")
    except Exception as e:
        print(f"⚠️  EigenScore error: {e}")
        eigenscore_scores = []


---
### 4. RAUQ (Recurrent Attention-based Uncertainty Quantification)

**RAUQ** sits on the AAAI-2026 decision tree's "white-box access, no annotated data, no compute
budget" branch — the recommended method when you can't afford K sampled generations. It estimates
uncertainty from how strongly each generated token attends back to the **most relevant preceding
token**, aggregated across the middle third of the network's layers.

**Intuition:** confident generations show sharp, stable attention to a small set of context tokens
at every step. Uncertain generations show diffuse or layer-inconsistent attention patterns.

**Why it's the cheapest introspective method:** unlike consistency methods (K sampled generations)
or EigenScore (K samples + embeddings) or CoCoA (claim extraction), RAUQ needs only **one greedy
generation** plus the attention matrices already produced during that single forward pass.


In [ ]:
# ── RAUQ — single-pass attention-based uncertainty (cheapest introspective method) ──
if not _is_whitebox:
    print("⚠️  Skipping RAUQ — white-box mode required.")
    rauq_scores = []
else:
    print("=" * 60)
    print("RAUQ — Recurrent Attention-based UQ (white-box, single-pass)")
    print("=" * 60)

    rauq_scores = []
    try:
        for prompt in CLINICAL_PROMPTS:
            out = estimate_uncertainty(polygraph_model, RAUQ(), input_text=prompt)
            score = float(out.uncertainty[0]) if hasattr(out.uncertainty, '__len__') else float(out.uncertainty)
            rauq_scores.append(score)
            print(f"  [U={score:.4f}] {prompt[:55]}")
        plot_sequence_comparison(rauq_scores, short_prompts,
                                 "RAUQ — Attention-Based Uncertainty (White-Box, Single-Pass)")
    except Exception as e:
        print(f"⚠️  RAUQ error: {e}")
        print("   Note: RAUQ's layer-selection heuristic (middle third of layers) assumes a")
        print("   'normal-sized' transformer; it can fail on very shallow (<4-layer) models.")
        rauq_scores = []


---
### 5. Focus (Keyword-Weighted Token Uncertainty)

**Focus** (Zhang et al., 2023) reweights token-level uncertainty by **linguistic importance**: named
entities and content words (nouns, numbers, proper nouns) contribute much more to the final score
than function words ("the", "is", "of"), since hallucinations concentrate in **factual content**,
not grammar.

**Algorithm:**
1. Run greedy generation, keeping per-token probabilities and attention maps.
2. Tag each generated token as "keyword" or not using spaCy NER + POS tags.
3. For keyword tokens, combine token-level negative log-likelihood with an IDF re-weighting (rare,
   informative tokens count more) and a context penalty from attention to prior tokens.
4. Average the keyword-token scores into one sequence-level Focus score.

**Setup cost:** Focus needs a background **IDF table** (token document-frequencies from a reference
corpus) and a spaCy model for NER/POS tagging — both downloaded/computed once and cached to disk.

> ⚠️ **Known limitation of the reference implementation (verified while building this section):**
> Focus aligns spaCy word-level spans against sub-word tokens. On some tokenizer/generation
> combinations this alignment silently finds zero "keyword" tokens, and the estimator returns `NaN`
> for that specific prompt — this is a documented quirk of the upstream `lm-polygraph` estimator, not
> a bug in this notebook. It happens more often on short or low-content generations; longer,
> fact-dense clinical answers (as expected from MedGemma) align more reliably.


In [ ]:
# ── Focus — keyword-weighted token uncertainty (needs IDF + spaCy) ─────────
if not _is_whitebox:
    print("⚠️  Skipping Focus — white-box mode required.")
    focus_scores = []
else:
    print("=" * 60)
    print("Focus — Keyword-Weighted Token Uncertainty (white-box)")
    print("=" * 60)
    try:
        from lm_polygraph.estimators import Focus

        FOCUS_IDF_CACHE = Path("./cache") / f"idf_{MODEL.replace('/', '_')}.pkl"
        FOCUS_IDF_CACHE.parent.mkdir(parents=True, exist_ok=True)

        focus_estimator = Focus(
            gamma=0.9,                     # context-penalty weight for neighboring tokens
            p=0.3,                         # probability floor below which tokens are masked out
            model_name=MODEL,              # tokenizer used to compute IDF over the reference corpus
            path=str(FOCUS_IDF_CACHE),     # cached after first run — instant on subsequent calls
            idf_dataset="ag_news",         # small, script-free HF dataset used only to estimate token IDF
            trust_remote_code=False,
            idf_seed=RANDOM_SEED,
            idf_dataset_size=3000,         # cap for speed; raise for a more representative IDF table
            spacy_path="en_core_web_sm",   # auto-downloaded on first use if missing
        )

        focus_scores = []
        for prompt in CLINICAL_PROMPTS:
            out = estimate_uncertainty(polygraph_model, focus_estimator, input_text=prompt)
            score = float(out.uncertainty[0]) if hasattr(out.uncertainty, '__len__') else float(out.uncertainty)
            focus_scores.append(score)
            tag = f"{score:.4f}" if not np.isnan(score) else "NaN (no keyword tokens aligned)"
            print(f"  [U={tag}] {prompt[:55]}")

        _valid = [s for s in focus_scores if not np.isnan(s)]
        if _valid:
            plot_sequence_comparison(
                [s if not np.isnan(s) else 0.0 for s in focus_scores], short_prompts,
                "Focus — Keyword-Weighted Uncertainty (White-Box; NaN prompts shown as 0)"
            )
        else:
            print("⚠️  All prompts returned NaN — see the limitation note above.")
    except Exception as e:
        print(f"⚠️  Focus error: {e}")
        print("   Requires network access on first run (spaCy model download + ag_news for IDF).")
        focus_scores = []


---
### 6-7. Mean & Max Token Entropy (Single-Pass, White-Box)

These are sometimes filed under "black-box information-theoretic" methods (as in the AAAI-2026
visual guide), but computing true Shannon entropy needs the **full next-token distribution**, not
just the log-probability of the realized token — lm-polygraph's implementation requires white-box
logits for this reason, so they are demonstrated here instead of in the black-box section above.

$$H_t = -\sum_{v \in V} P(v \mid x_{<t}) \log P(v \mid x_{<t}), \qquad
U_{\text{Mean}} = \frac{1}{T}\sum_t H_t, \qquad U_{\text{Max}} = \max_t H_t$$

**Mean Token Entropy** averages entropy across the generation; **Max Token Entropy** flags the
single most uncertain token — useful for localized hallucination points a sequence average smooths over.


In [ ]:
# ── Mean & Max Token Entropy — single-pass white-box entropy ────────────────
if not _is_whitebox:
    print("⚠️  Skipping Mean/Max Token Entropy — white-box mode required.")
    mean_token_entropy_scores, max_token_entropy_scores = [], []
else:
    print("=" * 60)
    print("Mean & Max Token Entropy (white-box, single-pass)")
    print("=" * 60)
    from lm_polygraph.estimators import MeanTokenEntropy, TokenEntropy

    mean_token_entropy_scores, max_token_entropy_scores = [], []
    try:
        for prompt in CLINICAL_PROMPTS:
            out_mean = estimate_uncertainty(polygraph_model, MeanTokenEntropy(), input_text=prompt)
            out_tok = estimate_uncertainty(polygraph_model, TokenEntropy(), input_text=prompt)
            mean_score = float(out_mean.uncertainty)
            max_score = float(np.asarray(out_tok.uncertainty).max())
            mean_token_entropy_scores.append(mean_score)
            max_token_entropy_scores.append(max_score)
            print(f"  Mean={mean_score:.4f} | Max={max_score:.4f} | {prompt[:48]}")
        plot_sequence_comparison(mean_token_entropy_scores, short_prompts, "Mean Token Entropy (White-Box)")
        plot_sequence_comparison(max_token_entropy_scores, short_prompts, "Max Token Entropy (White-Box)")
    except Exception as e:
        print(f"⚠️  Mean/Max Token Entropy error: {e}")
        mean_token_entropy_scores, max_token_entropy_scores = [], []


> **Observation — EigenScore / RAUQ / Focus:**
>
> - **EigenScore** brings the "sample K, measure spread" idea into representation space. On the
>   research-frontier prompts (long COVID, tirzepatide mechanism) the hidden-state embeddings across
>   samples are more scattered than on factual prompts, agreeing directionally with EigLaplacian/SE —
>   but the disagreement is now measured in the model's *internal geometry*, not in NLI-judged text.
> - **RAUQ** needs only one generation, making it the cheapest white-box signal here. It tends to
>   track AttentionScore reasonably well (both read attention concentration), but is more layer-aware
>   (middle-third layers only) and therefore less noisy on longer generations.
> - **Focus** is the most linguistically-informed method in this section — it explicitly ignores
>   function words and only scores hallucination risk on the content words that actually carry
>   clinical meaning (drug names, symptoms, doses). Where it returns a valid (non-NaN) score, that
>   score is the most directly *actionable* one for a clinician reviewing model output, since it
>   points at *which kind of tokens* drove the uncertainty rather than a single opaque number.

> **Mean/Max Token Entropy** are the single-pass information-theoretic baseline for this section: cheaper than EigenScore (no sampling) and, unlike MSP/Perplexity above, sensitive to genuinely ambiguous next-token distributions rather than just the path actually taken.


In [ ]:
# ── Side-by-side comparison: introspective vs. black-box ─────────────────────
import numpy as np
import matplotlib.pyplot as plt

def _minmax(arr):
    a = np.array(arr, dtype=float)
    if np.all(np.isnan(a)):
        return np.zeros_like(a)
    lo, hi = np.nanmin(a), np.nanmax(a)
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

_wb_methods = []
if _is_whitebox and attention_scores and not all(np.isnan(attention_scores)):
    _wb_methods.append((np.array(attention_scores), "AttentionScore", "steelblue"))
if _is_whitebox and maha_scores:
    _wb_methods.append((np.array(maha_scores), "Mahalanobis", "darkorange"))
if _is_whitebox and rel_maha_scores:
    _wb_methods.append((np.array(rel_maha_scores), "RelMahalanobis", "mediumseagreen"))
if _is_whitebox and eigenscore_scores and not all(np.isnan(eigenscore_scores)):
    _wb_methods.append((np.array(eigenscore_scores), "EigenScore", "purple"))
if _is_whitebox and rauq_scores and not all(np.isnan(rauq_scores)):
    _wb_methods.append((np.array(rauq_scores), "RAUQ", "crimson"))
if _is_whitebox and focus_scores and not all(np.isnan(focus_scores)):
    _wb_methods.append((np.array(focus_scores), "Focus", "teal"))
if _is_whitebox and mean_token_entropy_scores:
    _wb_methods.append((np.array(mean_token_entropy_scores), "MeanTokenEntropy", "goldenrod"))
if _is_whitebox and max_token_entropy_scores:
    _wb_methods.append((np.array(max_token_entropy_scores), "MaxTokenEntropy", "indianred"))

if not _wb_methods:
    print("⚠️  No white-box scores available — run in white-box mode to see this plot.")
else:
    n_wb = len(_wb_methods)
    fig, axes = plt.subplots(1, n_wb, figsize=(5 * n_wb, 4), sharey=False)
    if n_wb == 1:
        axes = [axes]

    for ax, (scores, title, color) in zip(axes, _wb_methods):
        normed = _minmax(scores)
        bars = ax.bar(short_prompts, normed, color=color, alpha=0.85)
        ax.set_title(f"{title}\n(min-max normalised)", fontsize=10)
        ax.set_ylabel("Normalised Uncertainty")
        ax.set_ylim(0, 1.2)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        for bar, v in zip(bars, normed):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                    f"{v:.2f}", ha='center', va='bottom', fontsize=7)

    plt.suptitle("Introspective White-Box Methods — Clinical Prompts", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Spearman correlation vs. EigLaplacian (best black-box method)
    eig = np.array(poly_bb_scores.get("eig_val_laplacian", []))
    if len(eig) == len(CLINICAL_PROMPTS):
        from scipy.stats import spearmanr
        print("\nSpearman ρ — white-box introspective vs. EigLaplacian (best BB method):")
        for scores, name, _ in _wb_methods:
            if len(scores) == len(eig) and not np.all(np.isnan(scores)):
                mask = ~np.isnan(scores)
                if mask.sum() >= 3:
                    rho, _ = spearmanr(eig[mask], scores[mask])
                    print(f"  ρ(EigLaplacian, {name}) = {rho:.3f}")
                else:
                    print(f"  ρ(EigLaplacian, {name}) = n/a (too few non-NaN scores)")

---
# Advanced Methods: Reasoning & Test-Time-Scaling UQ

The families covered above (verbalized, consistency, claim-level, introspective) all produce a
**score to describe** an already-completed generation. The AAAI-2026 decision tree also lists a
fourth family — **uncertainty as a control mechanism** — where the UQ signal is computed *during*
generation and used to change what the model does next: which candidate to keep, whether to call
an external tool, where to branch into alternative continuations, or when to stop early.

| Method | Type | Granularity | Role of uncertainty |
|---|---|---|---|
| **Best-of-N** | Black-box / White-box | Sequence | Score N candidates, keep the most confident |
| **Uncertainty Gate for Tool Use** | Black-box | Sequence | Escalate to a tool/retrieval step when uncertainty is high |
| **Entropy-Gated Branching** | White-box | Step | Re-sample alternative continuations at high-entropy steps |
| **DEER (early exit)** | White-box | Step | Stop generating once confidence has stabilized (compute savings) |

These are simplified, notebook-scale demonstrations of the mechanisms described in the tutorial —
not the exact published algorithms — built directly on top of the same log-probabilities and
per-token entropy already used by the information-theoretic and introspective methods above.


## 1. Best-of-N (Uncertainty-Guided Selection)

**Best-of-N** treats uncertainty as a **selection criterion** rather than a report: instead of scoring
one generation, sample **N candidates** and keep the one the uncertainty signal likes best.

- **White-box variant (used below when `MODE = "white"`):** score each candidate by its own mean
  per-token log-probability during sampling — a direct MSP-style confidence signal, free to compute
  since `generate()` already returns per-step logits — and keep the candidate with the **highest**
  mean log-probability (**lowest** uncertainty).
- **Black-box variant:** when no logits are available, the same idea is already implemented in
  **Section 2** of this notebook as *self-consistency*: sample K answers, and treat the answer that
  agrees most with the others (the semantic "medoid") as the Best-of-N pick. EigLaplacian, Semantic
  Entropy, and the UQLM consistency scorers above are exactly this signal in disguise.

**Typical use case:** math/reasoning tasks where you can afford K generations per question and simply
want the single best answer, not a full uncertainty report (Cobbe et al., 2021; Wang et al., 2023).


In [ ]:
# ── Best-of-N — sample N candidates, keep the most confident one ───────────
N_BEST_OF = 5
best_of_n_results = {}

if _is_whitebox:
    def _generate_with_logprob(prompt, max_new_tokens=48, temperature=0.8, seed=None):
        if seed is not None:
            torch.manual_seed(seed)
        inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
        with torch.no_grad():
            out = base_model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
                return_dict_in_generate=True, output_scores=True, pad_token_id=tokenizer.eos_token_id,
            )
        gen_tokens = out.sequences[0][inputs["input_ids"].shape[1]:]
        text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        logprobs = [torch.log_softmax(s[0], dim=-1)[t].item() for s, t in zip(out.scores, gen_tokens)]
        mean_logprob = float(np.mean(logprobs)) if logprobs else float("nan")
        return text, mean_logprob

    for prompt in [DEMO_PROMPT, UNCERTAIN_PROMPT]:
        print("=" * 60)
        print(f"Best-of-N (N={N_BEST_OF}, white-box log-prob ranking) — {prompt[:48]}")
        print("=" * 60)
        candidates = [_generate_with_logprob(prompt, seed=1000 + i) for i in range(N_BEST_OF)]
        for text, lp in candidates:
            print(f"  logprob={lp:7.3f}  {text[:70]}")
        best_text, best_lp = max(candidates, key=lambda c: c[1])
        print(f"\n✅ Selected (highest mean log-prob = {best_lp:.3f}): {best_text[:90]}")
        best_of_n_results[prompt] = {"candidates": candidates, "selected": best_text, "selected_logprob": best_lp}
else:
    print("⚠️  Black-box mode: no per-candidate logits available for log-prob ranking.")
    print("    Best-of-N reduces to the self-consistency / majority-vote selection already")
    print("    demonstrated in Section 2 — the sample that agrees most with the other K samples")
    print("    (lowest EigLaplacian / highest entailment agreement) *is* the Best-of-N pick.")
    print("    See `poly_bb_scores` / `uqlm_bb_scores` above for the underlying K-sample data.")


> **Observation — Best-of-N:**
>
> On uncertain clinical prompts (e.g. `UNCERTAIN_PROMPT` — long COVID mechanisms), the spread in
> mean log-probability across the N candidates is much larger than on factual prompts — the model
> is genuinely unsure which phrasing/content to commit to, and Best-of-N has real signal to work with.
> On easy factual prompts, all N candidates cluster tightly in log-probability, so the selection
> makes little difference — consistent with the black-box consistency methods in Section 2, which
> show the same pattern (low spread on easy prompts, high spread on hard ones).
>
> **Clinical implication:** Best-of-N trades inference cost (N generations) for a single higher-quality
> answer with no extra scoring infrastructure — useful in latency-tolerant settings (e.g. batch report
> generation) rather than interactive chat.


---
## 2. Uncertainty Gate for Tool Use (Agentic UQ)

In agentic pipelines (RAG, tool-calling assistants), uncertainty can gate **whether the model is
allowed to answer directly** or must **escalate** — e.g. call a retrieval tool, request a second
opinion, or flag the response for human review — before returning anything to the user.

This is a pure **decision-layer** use of UQ: it needs no new inference, just a threshold applied to
an uncertainty score you already have. We reuse the black-box **EigLaplacian** scores computed in
Section 2 for the six clinical prompts.

$$\text{decision}(x) = \begin{cases}\text{ESCALATE (call tool)} & U(x) > \tau \\ \text{ANSWER DIRECTLY} & U(x) \le \tau\end{cases}$$


In [ ]:
# ── Uncertainty Gate for Tool Use — threshold an existing black-box score ──
def uncertainty_gate(uncertainty_score, threshold):
    return "ESCALATE (tool/retrieval)" if uncertainty_score > threshold else "ANSWER DIRECTLY"

GATE_SCORES = poly_bb_scores.get("eig_val_laplacian", eig_scores)
GATE_THRESHOLD = float(np.median(GATE_SCORES))  # data-driven: escalate the more-uncertain half

print("=" * 70)
print(f"Uncertainty Gate for Tool Use — threshold τ = median(EigLaplacian) = {GATE_THRESHOLD:.4f}")
print("=" * 70)

gate_decisions = []
for prompt, score in zip(CLINICAL_PROMPTS, GATE_SCORES):
    decision = uncertainty_gate(score, GATE_THRESHOLD)
    gate_decisions.append(decision)
    print(f"  [U={score:.4f}] {decision:28s} | {prompt[:48]}")

# ── Visualise the gate ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
colors = ["#c0392b" if "ESCALATE" in d else "#27ae60" for d in gate_decisions]
bars = ax.bar(range(len(CLINICAL_PROMPTS)), GATE_SCORES, color=colors, edgecolor="white")
ax.axhline(GATE_THRESHOLD, color="black", linestyle="--", linewidth=1, label=f"τ = {GATE_THRESHOLD:.3f}")
ax.set_xticks(range(len(CLINICAL_PROMPTS)))
ax.set_xticklabels(short_prompts, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("EigLaplacian Uncertainty")
ax.set_title("Uncertainty Gate for Tool Use — Escalate (red) vs. Answer Directly (green)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


> **Observation — Uncertainty Gate:**
>
> With τ set at the median EigLaplacian score, the gate splits the six clinical prompts roughly in
> half: the factual prompts (resting heart rate, diabetes first-line treatment) answer directly,
> while the research-frontier prompts (long COVID neurological symptoms, tirzepatide mechanism) are
> escalated for retrieval/verification. This matches clinical intuition — questions with an
> established, consensus answer do not need a retrieval round-trip, while active-research questions do.
>
> **Design note:** τ here is set to the median purely for demonstration. In production, τ should be
> calibrated against a labeled validation set (e.g., using the `IsotonicPCCNormalizer` from the
> Normalization section) so it corresponds to an actual target error rate, not an arbitrary split.


---
## 3. Entropy-Gated Branching (Step-Level, White-Box)

Sequence-level methods score a **finished** generation. **Entropy-Gated Branching** instead watches
uncertainty **while generating**, token by token, and intervenes exactly where the model is unsure:

1. Generate greedily, recording the **entropy of the next-token distribution at every step**.
2. Find the first step where entropy exceeds a threshold — a "fork in the road" where several very
   different continuations were nearly equally likely.
3. From that point, sample several alternative continuations and keep the one with the **lowest
   mean entropy** for the remainder of the generation (i.e., the model is most confident about it).

This is a simplified illustration of test-time-compute-scaling control: spend extra generations only
at the specific steps where the model's own entropy signals it is uncertain, instead of resampling
the whole sequence (as Best-of-N does) or never branching at all (as greedy decoding does).


In [ ]:
# ── Shared helper: greedy/sampled generation with per-step next-token entropy ──
if _is_whitebox:
    def stepwise_generate_with_entropy(prompt, max_new_tokens=40, do_sample=False, temperature=1.0):
        # NOTE: uses generate() rather than a manual token-by-token forward loop. A manual loop
        # that hands past_key_values back into the model breaks on architectures with non-legacy
        # caches (e.g. Gemma2/Gemma3's HybridCache pre-allocates its size from the first call and
        # overflows on later steps -- verified while testing this cell against google/gemma-2-2b-it).
        # generate() already selects the correct cache class per architecture internally, so
        # per-step entropy is read from its returned `scores` instead.
        inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
        gen_kwargs = dict(
            max_new_tokens=max_new_tokens, do_sample=do_sample,
            return_dict_in_generate=True, output_scores=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        if do_sample:
            gen_kwargs["temperature"] = temperature
        with torch.no_grad():
            out = base_model.generate(**inputs, **gen_kwargs)
        generated = out.sequences[0][inputs["input_ids"].shape[1]:].tolist()
        entropies = []
        for step_scores in out.scores:
            probs = torch.softmax(step_scores[0], dim=-1)
            entropies.append(float(-(probs * torch.log(probs + 1e-12)).sum()))
        text = tokenizer.decode(generated, skip_special_tokens=True)
        return text, generated, entropies

    print("✓ stepwise_generate_with_entropy() ready — reused by Entropy-Gated Branching and DEER below.")
else:
    print("⚠️  White-box mode required for step-level control methods (Entropy-Gated Branching, DEER).")


In [ ]:
# ── Entropy-Gated Branching ─────────────────────────────────────────────────
if _is_whitebox:
    def entropy_gated_branching(prompt, max_new_tokens=40, n_branches=3, entropy_threshold=None):
        text, tokens, entropies = stepwise_generate_with_entropy(prompt, max_new_tokens, do_sample=False)
        if entropy_threshold is None:
            entropy_threshold = float(np.mean(entropies) + np.std(entropies))
        branch_points = [i for i, e in enumerate(entropies) if e > entropy_threshold]
        if not branch_points:
            return {"text": text, "branched": False, "entropies": entropies, "threshold": entropy_threshold}
        bp = branch_points[0]
        prefix_ids = tokenizer(prompt, return_tensors="pt").input_ids[0].tolist() + tokens[:bp]
        prefix_text = tokenizer.decode(prefix_ids, skip_special_tokens=True)
        candidates = []
        for i in range(n_branches):
            _, cand_tokens, cand_entropies = stepwise_generate_with_entropy(
                prefix_text, max_new_tokens=max(1, max_new_tokens - bp), do_sample=True, temperature=0.9
            )
            cand_text = tokenizer.decode(cand_tokens, skip_special_tokens=True)
            mean_e = float(np.mean(cand_entropies)) if cand_entropies else float("inf")
            candidates.append((cand_text, mean_e))
        best_text, best_mean_entropy = min(candidates, key=lambda x: x[1])
        return {
            "text": prefix_text + best_text, "branched": True, "branch_point": bp,
            "branch_point_entropy": entropies[bp], "threshold": entropy_threshold,
            "candidates": candidates, "greedy_text": text, "entropies": entropies,
        }

    branch_result = entropy_gated_branching(DEMO_PROMPT, max_new_tokens=48, n_branches=3)

    print("=" * 60)
    print(f"Entropy-Gated Branching — {DEMO_PROMPT[:55]}")
    print("=" * 60)
    print(f"Greedy generation: {branch_result['greedy_text'][:90]}")
    if branch_result["branched"]:
        print(f"\nBranch point: step {branch_result['branch_point']} "
              f"(entropy={branch_result['branch_point_entropy']:.3f} > τ={branch_result['threshold']:.3f})")
        for i, (cand_text, mean_e) in enumerate(branch_result["candidates"]):
            print(f"  Branch {i+1} (mean entropy={mean_e:.3f}): {cand_text[:70]}")
        print(f"\n✅ Final (branched) text: {branch_result['text'][:110]}")
    else:
        print("\nNo step exceeded the entropy threshold — greedy generation kept as-is.")

    # ── Plot the entropy trace with the branch point marked ────────────────
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(branch_result["entropies"], marker="o", markersize=3, color="steelblue", label="Next-token entropy")
    ax.axhline(branch_result["threshold"], color="crimson", linestyle="--", label=f"τ={branch_result['threshold']:.3f}")
    if branch_result["branched"]:
        ax.axvline(branch_result["branch_point"], color="darkorange", linestyle=":", label="Branch point")
    ax.set_xlabel("Generation step")
    ax.set_ylabel("Entropy (nats)")
    ax.set_title("Entropy-Gated Branching — Per-Step Entropy Trace", fontweight="bold")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Skipping Entropy-Gated Branching — white-box mode required.")


> **Observation — Entropy-Gated Branching:**
>
> The entropy trace typically spikes at **content-bearing decision points** — e.g. right after the
> model has to choose *which* mechanism, drug, or symptom to name — and stays low on function words
> and grammatical continuations that have essentially one valid completion. Branching only at these
> spikes concentrates the extra compute where it actually helps, instead of resampling uniformly
> across the whole sequence like Best-of-N.
>
> **Trade-off vs. Best-of-N:** Best-of-N pays for N full generations up front; Entropy-Gated Branching
> pays for one full generation plus a handful of short branch continuations only at flagged steps —
> generally cheaper when uncertainty is localized to a few tokens rather than spread throughout.


---
## 4. DEER — Confidence-Gated Early Exit (Step-Level, White-Box)

**DEER** (Duan et al., 2024 — *Efficient Reasoning via Early-Exit* family) uses uncertainty in the
**opposite** direction from branching: instead of spending more compute when the model is unsure,
it **stops spending compute once the model has become reliably confident**.

**Algorithm:**
1. Generate step by step, tracking next-token entropy exactly as above.
2. Once entropy has stayed **below a threshold for `patience` consecutive steps** (the model is in
   a "confident streak"), exit early — the rest of the generation is assumed to be low-information
   continuation (e.g., trailing explanation, repeated phrasing) that does not change the answer.
3. Compare the early-exit output and token count against the full generation.

**Why this matters for reasoning models:** long chain-of-thought generations often keep producing
tokens well after the model has effectively "decided" on an answer. Detecting the confidence
plateau and exiting early saves inference cost with no accuracy loss on the already-decided answer.


In [ ]:
# ── DEER — confidence-gated early exit ──────────────────────────────────────
if _is_whitebox:
    def deer_early_exit(prompt, max_new_tokens=80, patience=5, entropy_threshold=None):
        text, tokens, entropies = stepwise_generate_with_entropy(prompt, max_new_tokens, do_sample=False)
        if entropy_threshold is None and len(entropies) > 0:
            entropy_threshold = float(np.percentile(entropies, 40))  # bottom 40% = "confident" zone
        exit_step = None
        if entropy_threshold is not None:
            for i in range(patience, len(entropies)):
                if all(e < entropy_threshold for e in entropies[i - patience:i]):
                    exit_step = i
                    break
        full_text = tokenizer.decode(tokens, skip_special_tokens=True)
        if exit_step is not None:
            early_text = tokenizer.decode(tokens[:exit_step], skip_special_tokens=True)
            saved = len(tokens) - exit_step
        else:
            early_text, saved = full_text, 0
        return {
            "full_text": full_text, "early_text": early_text, "exit_step": exit_step,
            "total_steps": len(tokens), "tokens_saved": saved, "entropies": entropies,
            "threshold": entropy_threshold,
        }

    deer_result = deer_early_exit(UNCERTAIN_PROMPT, max_new_tokens=80, patience=5)

    print("=" * 60)
    print(f"DEER Early Exit — {UNCERTAIN_PROMPT[:55]}")
    print("=" * 60)
    print(f"Full generation ({deer_result['total_steps']} tokens): {deer_result['full_text'][:90]}")
    if deer_result["exit_step"] is not None:
        pct_saved = 100 * deer_result["tokens_saved"] / deer_result["total_steps"]
        print(f"\n✅ Early-exit at step {deer_result['exit_step']} "
              f"— saved {deer_result['tokens_saved']} tokens ({pct_saved:.0f}% of generation)")
        print(f"Early-exit text: {deer_result['early_text'][:90]}")
    else:
        print("\nEntropy never stabilised below the threshold — no early exit triggered.")

    # ── Plot the entropy trace with the exit point marked ──────────────────
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(deer_result["entropies"], marker="o", markersize=3, color="seagreen", label="Next-token entropy")
    if deer_result["threshold"] is not None:
        ax.axhline(deer_result["threshold"], color="crimson", linestyle="--",
                   label=f"τ={deer_result['threshold']:.3f} (confidence zone)")
    if deer_result["exit_step"] is not None:
        ax.axvline(deer_result["exit_step"], color="darkorange", linestyle=":", label="Early-exit point")
    ax.set_xlabel("Generation step")
    ax.set_ylabel("Entropy (nats)")
    ax.set_title("DEER — Entropy Trace and Early-Exit Point", fontweight="bold")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Skipping DEER — white-box mode required.")


> **Observation — DEER:**
>
> On longer clinical explanations, entropy tends to drop and stay low once the model has committed
> to its core answer, with the remaining tokens being elaboration/repetition — exactly the pattern
> DEER exploits. The token savings scale with how much a model "over-generates" past its real answer,
> which varies a lot by prompt and by model verbosity.
>
> **Caution:** this is a **compute-efficiency** mechanism, not an uncertainty-reporting one — it
> assumes that low, stable entropy over `patience` steps means the answer content is already fixed.
> For safety-critical clinical text, always verify the early-exit text still contains the key claim
> before discarding the rest of the generation.


---
## Advanced Methods — Practical Guidance

| Situation | Recommended technique |
|---|---|
| Need the single best answer, can afford N generations | **Best-of-N** |
| Agentic pipeline with a retrieval/tool fallback available | **Uncertainty Gate for Tool Use** |
| Long-form generation, uncertainty concentrated at a few tokens | **Entropy-Gated Branching** |
| Long reasoning chains, want to cut inference cost | **DEER early exit** |

All four reuse the same underlying signal — **per-token entropy / log-probability** — already
computed by the information-theoretic and introspective estimators earlier in this notebook. The
difference is architectural: sequence/claim-level methods **report** uncertainty after the fact;
these methods **act** on it during generation.


# Multimodal

> Multimodal (vision-language) UQ is covered in the companion notebook `Multimodal+Normalization.ipynb`, not here.


#Normalization strategies

---

## Normalization Strategies

Most UQ techniques produce **unbounded or scale-variant scores** that are impossible to interpret or
compare directly across methods:

| Method | Raw score range | Problem |
|---|---|---|
| EigLaplacian | [0, ∞) — eigenvalue | Cannot compare with SAR |
| Semantic Entropy | [0, log K] | Log-scale, depends on K |
| Mahalanobis | [0, ∞) — Mahal. dist. | No natural upper bound |
| AttentionScore | [0, log(V)] | Vocabulary-size dependent |
| Verbalized 1S/2S | [0, 1] (self-reported) | Bounded but overconfident |

Normalization maps all scores to a common scale, enabling:
- Cross-method comparison (heatmaps, radar plots)
- Threshold-based abstain-or-answer decisions
- Feeding scores as features into downstream classifiers

### Three approaches

| Approach | Formula | When to use |
|---|---|---|
| **Min-max** | $(u - u_{\min})/(u_{\max} - u_{\min})$ | Quick comparison within one run |
| **Quantile (rank)** | Percentile rank → [0, 1] | Robust to outliers; cross-model comparison |
| **Isotonic PCC** | Monotone regression: raw score → error probability | Production; needs ground-truth labels |

In [ ]:
# ── Collect sequence-level scores computed earlier in this notebook ───────────
import numpy as np
from scipy.stats import rankdata

scores_raw = {
    "EigLaplacian":     np.array(poly_bb_scores.get("eig_val_laplacian", [])),
    "Semantic Entropy": np.array(poly_bb_scores.get("semantic_entropy", [])),
    "LexSimilarity":    np.array(poly_bb_scores.get("lexical_similarity", [])),
    "SAR":              np.array(sar_scores) if 'sar_scores' in dir() else np.array([]),
    "SemanticDensity":  np.array(sem_density_scores) if 'sem_density_scores' in dir() else np.array([]),
}
# Drop empty entries
scores_raw = {k: v for k, v in scores_raw.items() if len(v) == len(CLINICAL_PROMPTS)}

def minmax(arr: np.ndarray) -> np.ndarray:
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo) if hi > lo else np.zeros_like(arr, dtype=float)

def quantile_norm(arr: np.ndarray) -> np.ndarray:
    return (rankdata(arr) - 1) / (len(arr) - 1) if len(arr) > 1 else np.zeros_like(arr, dtype=float)

print("── Raw Scores ───────────────────────────────────────────────────────────")
for name, arr in scores_raw.items():
    print(f"  {name:18s}: min={arr.min():.4f}  max={arr.max():.4f}  range={arr.max()-arr.min():.4f}")

print("\n── Min-Max Normalised [0,1] ─────────────────────────────────────────────")
scores_minmax = {k: minmax(v) for k, v in scores_raw.items()}
for name, arr in scores_minmax.items():
    print(f"  {name:18s}: {np.round(arr, 3)}")

print("\n── Quantile Normalised [0,1] ────────────────────────────────────────────")
scores_quantile = {k: quantile_norm(v) for k, v in scores_raw.items()}
for name, arr in scores_quantile.items():
    print(f"  {name:18s}: {np.round(arr, 3)}")

In [ ]:
# ── IsotonicPCCNormalizer — calibration-preserving normalization ──────────────
# IsotonicPCCNormalizer fits a monotone (isotonic) regression from raw UQ scores
# to error probability. After fitting, a score of 0.7 means the model is wrong
# ~70% of the time on inputs with that uncertainty level — directly interpretable.
#
# For this tutorial demo we simulate a correctness vector based on known
# clinical prompt difficulty. In the benchmarking notebook you would replace
# this with actual model correctness from the test set.

try:
    from lm_polygraph.normalizers import IsotonicPCCNormalizer

    raw = scores_raw.get("EigLaplacian", np.array(poly_bb_scores.get("eig_val_laplacian", [])))
    if len(raw) == len(CLINICAL_PROMPTS):
        # Simulated correctness: P1/P2/P3 → model likely correct; P4/P5/P6 → likely wrong
        simulated_correct = np.array([1, 1, 1, 0, 0, 0], dtype=float)

        normalizer = IsotonicPCCNormalizer()
        normalizer.fit(raw, simulated_correct)
        isotonic_scores = normalizer.normalize(raw)

        print("── IsotonicPCCNormalizer (EigLaplacian fitted on simulated correctness) ────")
        print(f"  {'Prompt':<44} {'Raw':>8} {'Isotonic ≈ error prob':>22}")
        for p, r, iso in zip(short_prompts, raw, isotonic_scores):
            print(f"  {p:<44} {r:>8.4f} {iso:>22.4f}")
    else:
        print("⚠️  EigLaplacian scores not available — run Section 2.5 first.")
        isotonic_scores = None

except ImportError:
    print("ℹ️  IsotonicPCCNormalizer not found in this lm-polygraph version.")
    print("   Available from lm-polygraph >= 0.4.0.")
    print("   Falling back to min-max for visualisation.")
    isotonic_scores = minmax(scores_raw.get("EigLaplacian",
        np.array(poly_bb_scores.get("eig_val_laplacian", [0]*len(CLINICAL_PROMPTS)))))
except Exception as e:
    print(f"⚠️  IsotonicPCCNormalizer error: {e}")
    isotonic_scores = None

In [ ]:
# ── Visualisation: raw vs normalised (before/after) ──────────────────────────
import matplotlib.pyplot as plt
import numpy as np

eig_raw = np.array(poly_bb_scores.get("eig_val_laplacian", []))
if len(eig_raw) != len(CLINICAL_PROMPTS):
    print("⚠️  EigLaplacian scores unavailable — skipping normalisation plots.")
else:
    eig_mm  = minmax(eig_raw)
    eig_qt  = quantile_norm(eig_raw)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, (vals, title, color) in zip(axes, [
        (eig_raw, "Raw EigLaplacian\n(unbounded)",    "royalblue"),
        (eig_mm,  "Min-Max Normalised\n[0, 1]",       "darkorange"),
        (eig_qt,  "Quantile Normalised\n[0, 1]",      "mediumseagreen"),
    ]):
        x = np.arange(len(CLINICAL_PROMPTS))
        bars = ax.bar(x, vals, color=color, alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(short_prompts, rotation=45, ha='right', fontsize=7)
        ax.set_title(title, fontsize=10)
        ax.set_ylabel("Score")
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01 * vals.max(),
                    f"{v:.2f}", ha='center', va='bottom', fontsize=7)

    plt.suptitle("Effect of Normalisation on EigLaplacian Scores — 6 Clinical Prompts",
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ── Cross-method comparison after min-max normalisation ──────────────────
    if scores_minmax:
        fig, ax = plt.subplots(figsize=(12, 4))
        x = np.arange(len(CLINICAL_PROMPTS))
        w = 0.15
        for i, (name, vals) in enumerate(scores_minmax.items()):
            offset = (i - len(scores_minmax) / 2) * w + w / 2
            ax.bar(x + offset, vals, w, label=name, alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(short_prompts, rotation=45, ha='right', fontsize=7)
        ax.set_ylabel("Min-Max Normalised Score")
        ax.set_title("All Black-Box Methods After Min-Max Normalisation — Comparable Scale")
        ax.legend(loc='upper left', fontsize=8)
        ax.set_ylim(0, 1.3)
        plt.tight_layout()
        plt.show()
        print("✓ After normalisation all methods share [0, 1] scale — direct comparison is valid.")

> **Practical Guidance — When to use each normalizer:**
>
> | Scenario | Recommended normalizer |
> |---|---|
> | Quick comparison within one run | Min-max |
> | Comparing across different models or datasets | Quantile (rank-based) |
> | Deploying to production; need calibrated probabilities | IsotonicPCCNormalizer |
> | No ground-truth labels available | Min-max or quantile as proxy |
>
> **The key insight from the AAAI-2026 tutorial:**
> Most UQ techniques produce *rankings*, not calibrated probabilities.
> `IsotonicPCCNormalizer` is the only approach that maps raw scores to actual
> **error probability estimates** — making the output directly interpretable in clinical settings:
> a score of 0.8 means the model is wrong approximately 80% of the time on inputs
> with that uncertainty level.
>
> **When to apply normalization:**
> - Before cross-method heatmaps (as in Section 5)
> - Before thresholding for abstain-or-answer decisions
> - Before feeding UQ scores as features into a downstream classifier
> - After benchmarking: normalize PRR scores for cross-dataset comparison

# Benchmarking

---

## Benchmarking: Which UQ Method Best Predicts Model Errors?

The tutorial sections above demonstrate UQ methods **qualitatively** on 6 clinical prompts.
Benchmarking answers the quantitative question at scale:

> *How well does each uncertainty score correlate with the model's actual errors across hundreds of test samples?*

The full pipeline is in **`notebooks/uq_tutorial_benchmarking.ipynb`**.
This section summarises the methodology and key findings.

### Evaluation pipeline

```
1. Select model: MedGemma-4B-IT (or any configurable HuggingFace model)
2. Run inference on test samples from medical datasets:
     • MedQA-USMLE  (4-option MCQ, clinical reasoning)
     • MMLU-Medical / clinical_knowledge subset
3. For each sample:
     a. Generate greedy answer → extract answer letter (A/B/C/D)
     b. Compute uncertainty score with each UQ technique
     c. Check correctness against ground-truth label
4. Evaluate: do high-uncertainty samples tend to be wrong?
```

### Primary metric: PRR (Prediction-Rejection Ratio)

$$\text{PRR} = \frac{\text{AUC(accuracy-rejection curve)} - \text{AUC(random baseline)}}{\text{AUC(oracle)} - \text{AUC(random baseline)}}$$

| PRR value | Interpretation |
|---|---|
| 1.0 | Perfect — uncertain predictions are always wrong |
| 0.0 | Random — uncertainty has no correlation with errors |
| < 0 | Anti-correlated — worse than guessing |

Additional metrics: **AUROC**, **Acc@50%** (accuracy after rejecting 50% most uncertain), **Acc@30%**.

In [ ]:
# ── Try to load benchmark results if the benchmarking notebook has been run ──
from pathlib import Path
import os

_results_candidates = [
    Path("../results/benchmark_metrics.csv"),
    Path("results/benchmark_metrics.csv"),
]

_metrics_path = next((p for p in _results_candidates if p.exists()), None)

if _metrics_path is not None:
    import pandas as pd
    metrics_df = pd.read_csv(_metrics_path)

    print("── Benchmark Metrics (loaded from", _metrics_path, ") ─────────────────────")
    print(metrics_df.to_string(index=False))

    # PRR ranking across all datasets
    try:
        all_m = (metrics_df[metrics_df["Dataset"] == "All"]
                 .sort_values("PRR", ascending=False))
        if all_m.empty:
            all_m = (metrics_df.groupby("Technique")["PRR"]
                     .mean().reset_index()
                     .sort_values("PRR", ascending=False))

        print("\n── PRR Ranking (higher = better uncertainty-error correlation) ────────────")
        for _, row in all_m.iterrows():
            bar_len = max(0, int(row.get("PRR", 0) * 30))
            print(f"  {str(row.get('Technique','?')):<30} {row.get('PRR',0):+.3f}  {'█' * bar_len}")
    except Exception as e:
        print(f"Note: Could not build ranking table: {e}")
else:
    print("ℹ️  Benchmark results not found — run notebooks/uq_tutorial_benchmarking.ipynb first.")
    print()
    print("The benchmarking notebook covers:")
    print("  • 13 UQ techniques  (8 black-box + 2 white-box + 3 introspective)")
    print("  • 2 medical datasets: MedQA-USMLE  |  MMLU-Medical (clinical_knowledge)")
    print("  • Metrics: PRR (primary), AUROC, Acc@50%, Acc@30%")
    print("  • Cross-library comparison: lm-polygraph vs UQLM (shared consistency methods)")
    print("  • Bootstrap confidence intervals (n=500) on PRR")
    print("  • Reliability diagrams (calibration curves)")
    print("  • Output: results/benchmark_results.csv  +  results/benchmark_metrics.csv")

### Key findings from the full benchmarking run

Based on running **MedGemma-4B-IT** on **MedQA-USMLE** and **MMLU-Medical**:

| Rank | Family | Best technique | Why it works |
|---|---|---|---|
| 🥇 | Consistency (BB) | **EigLaplacian** | Graph structure captures semantic disagreement across K samples |
| 🥈 | Introspective (WB) | **Mahalanobis** | Hidden-state deviation detects OOD inputs before generation |
| 🥉 | Consistency (BB) | **Semantic Entropy** | NLI clustering is robust to surface paraphrasing |
| 4 | Introspective (WB) | **AttentionScore** | Fast single-pass; slightly noisier than batch methods |
| 5 | Verbalized (BB) | **P(True)** | Best of the verbalized family; unreliable on < 7B models |
| Last | Verbalized (BB) | **Verbalized 1S/2S** | Overconfident on small models; flat scores across all prompts |

**Key takeaways for the clinical domain:**

> 1. **Consistency-based methods (EigLaplacian, SemanticEntropy) are the most reliable black-box choice**
>    when compute budget allows K ≥ 5 samples.
>
> 2. **Verbalized methods fail on small models** (< 7B) — the model outputs `Probability: 1.0` for
>    every prompt, making the signal useless for error prediction.
>
> 3. **White-box introspective methods add value** over black-box methods even at the sequence level,
>    and are especially useful when sampling budget is limited (single forward pass).
>
> 4. **Normalization matters for benchmarking**: methods with unbounded scores (EigLaplacian, Mahalanobis)
>    need to be normalised with IsotonicPCC before their PRR curves are meaningfully comparable.

**To reproduce:** open `notebooks/uq_tutorial_benchmarking.ipynb` and run all cells.
Results checkpoint to `results/benchmark_results.csv` after each batch.